# 2 Parameter Sweep

Hyperparameter sweep for all ten encoders on 400 event + 400 noise files; selects the operating point used in the benchmark.

Paths are imported from `config.py` at the repository root.

In [ ]:
"""
MULTI-FILE PARAMETER SWEEP — ALL 10 ENCODERS (VECTORISED, WIDE RANGES)
=======================================================================
Sweeps key parameters for each encoder across a random sample of
N_SAMPLES event + N_SAMPLES noise files from the full dataset.

All encoders are fully or partially vectorised (numpy) for speed.

Encoders covered (all 10):
    Baselines:  [B1] Rate, [B2] Phase, [B3] TTFS
    Proposed:   [P1] Delta-Mod, [P2] PDE, [P3] ATDE, [P4] MASTE,
                [P5] ST-MW, [P6] ICDE, [P7] SFBE

Estimated runtime on Jetson AGX Orin (N_SAMPLES=200):
    Baselines (B1–B3):              ~10–15  min
    Proposed simple (P1–P4):        ~30–40  min
    Proposed spatial (P5–P7):       ~60–90  min
    Total:                          ~3–5 hours (recommended for paper)

"""

# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

import os
import json
import time
import random
import segyio
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, hilbert

# =============================================================================
# CONFIGURATION
# =============================================================================

EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
RESULTS_DIR = config.SWEEP_RESULTS_DIR

N_SAMPLES   = 400      # files per class — set to 50 for a quick test run
RANDOM_SEED = 42

FORCE_DT_MS     = 0.5
FILTER_LOW_HZ   = 50
FILTER_HIGH_HZ  = 250
WINDOW_START_MS = 0
WINDOW_END_MS   = 1000  # → 2000 samples at 0.5 ms
EVENT_START_MS  = 380.0
EVENT_END_MS    = 560.0

TARGET_SP_LO = 0.003   # 0.3%
TARGET_SP_HI = 0.015   # 1.5%

# ── DAS geometry — used by ST-MW and ICDE ────────────────────────────────────
TRACE_SPACING_M = 4.0

# ── SFBE sub-bands — fixed to seismic physics, not swept ─────────────────────
# Band 1: 50–100 Hz  (S-wave / surface wave dominant)
# Band 2: 100–150 Hz (mixed P/S arrivals)
# Band 3: 150–200 Hz (P-wave dominant)
# Band 4: 200–250 Hz (P-wave first arrivals / high-freq coda)
SFBE_BANDS = [(50, 100), (100, 150), (150, 200), (200, 250)]

os.makedirs(RESULTS_DIR, exist_ok=True)


# =============================================================================
# FILE SAMPLING
# =============================================================================

def sample_files(directory, n, seed):
    all_files = sorted([
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(('.sgy', '.segy'))
    ])
    if len(all_files) < n:
        print(f"   Only {len(all_files)} files in "
              f"{os.path.basename(directory)}, using all.")
        return all_files
    rng = random.Random(seed)
    return sorted(rng.sample(all_files, n))


# =============================================================================
# LOAD + PREPROCESS (single file)
# =============================================================================

def load_and_preprocess(filepath):
    """
    Returns dict:
        norm_data : unit-normalised waveform  float64 (n_tr, n_smp)
        norm_env  : unit-normalised envelope  float64 (n_tr, n_smp)
        filtered  : bandpass-filtered only    float64 (n_tr, n_smp)
        dt_ms     : sample interval in ms
    Returns None on load failure.
    """
    try:
        with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))],
                            dtype=np.float64)
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                if not (0.001 <= dt_ms <= 10.0):
                    dt_ms = FORCE_DT_MS
            except Exception:
                dt_ms = FORCE_DT_MS
    except Exception as ex:
        print(f"     {os.path.basename(filepath)}: {ex}")
        return None

    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ/nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ/nyq, 1e-6, 0.99)], btype='band')
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        filtered[i] = filtfilt(b, a, data[i])

    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(filtered.shape[1], int(WINDOW_END_MS / dt_ms))
    filtered = filtered[:, s:e]

    tmp = filtered - filtered.mean(axis=1, keepdims=True)
    mx  = np.where(np.max(np.abs(tmp), axis=1, keepdims=True) > 0,
                   np.max(np.abs(tmp), axis=1, keepdims=True), 1.0)
    norm_data = (tmp / mx + 1) / 2

    env_raw  = np.abs(hilbert(filtered, axis=1))
    mx_env   = np.where(np.max(env_raw, axis=1, keepdims=True) > 0,
                        np.max(env_raw, axis=1, keepdims=True), 1.0)
    env      = env_raw / mx_env
    tmp2     = env - env.mean(axis=1, keepdims=True)
    mx2      = np.where(np.max(np.abs(tmp2), axis=1, keepdims=True) > 0,
                        np.max(np.abs(tmp2), axis=1, keepdims=True), 1.0)
    norm_env = (tmp2 / mx2 + 1) / 2

    return {'norm_data': norm_data, 'norm_env': norm_env,
            'filtered': filtered, 'dt_ms': dt_ms}


def load_dataset(event_files, noise_files):
    print(f"\n  Preprocessing {len(event_files)} event files...")
    event_data = []
    for i, fp in enumerate(event_files):
        r = load_and_preprocess(fp)
        if r is not None:
            event_data.append(r)
        if (i+1) % 50 == 0:
            print(f"    {i+1}/{len(event_files)}")

    print(f"  Preprocessing {len(noise_files)} noise files...")
    noise_data = []
    for i, fp in enumerate(noise_files):
        r = load_and_preprocess(fp)
        if r is not None:
            noise_data.append(r)
        if (i+1) % 50 == 0:
            print(f"    {i+1}/{len(noise_files)}")

    print(f"  Loaded: {len(event_data)} event + {len(noise_data)} noise files")
    return event_data, noise_data


# =============================================================================
# METRICS
# =============================================================================

def compute_metrics(spikes, dt_ms):
    n_tr, n_smp = spikes.shape
    sp = float(spikes.mean())
    s  = max(0, int(EVENT_START_MS / dt_ms))
    e  = min(n_smp, int(EVENT_END_MS / dt_ms))
    density = spikes.mean(axis=0)
    in_ev   = float(density[s:e].mean()) if e > s else 0.0
    out_smp = np.concatenate([density[:s], density[e:]])
    min_d   = 1.0 / max(n_tr * n_smp, 1)
    out_ev  = float(max(out_smp.mean(), min_d)) if len(out_smp) > 0 else min_d
    snr     = in_ev / out_ev if out_ev > 0 else 0.0
    total   = float(spikes.sum())
    prec    = float(spikes[:, s:e].sum() / total) if total > 0 else 0.0
    return sp, snr, prec


def batch_metrics(spikes_list, dt_ms_list):
    sps, snrs, precs = [], [], []
    for spk, dt in zip(spikes_list, dt_ms_list):
        sp, snr, prec = compute_metrics(spk, dt)
        sps.append(sp); snrs.append(snr); precs.append(prec)
    return float(np.mean(sps)), float(np.mean(snrs)), float(np.mean(precs))


# =============================================================================
# VECTORISED ENCODERS
# All encoders operate on float64 (n_traces, n_samples) and return float32.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# BASELINES
# ─────────────────────────────────────────────────────────────────────────────

def enc_rate(data, dt, thr, seed=42):
    """B1: Rate — stochastic firing proportional to local amplitude."""
    rng    = np.random.default_rng(seed)
    spw    = max(1, int(1.0 / dt))
    n_tr, n_smp = data.shape
    n_bins = n_smp // spw
    trimmed = data[:, :n_bins * spw]
    binned  = trimmed.reshape(n_tr, n_bins, spw)
    amp     = np.mean(np.abs(binned), axis=2)
    mask    = amp > thr
    rand    = rng.random((n_tr, n_bins, spw))
    amp_c   = np.clip(amp, 0, 1)[:, :, np.newaxis]
    spikes  = (rand < amp_c) & mask[:, :, np.newaxis]
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    out[:, :n_bins * spw] = spikes.reshape(n_tr, n_bins * spw).astype(np.float32)
    return out


def enc_phase(data, dt, thr):
    """B2: Phase — amplitude encodes spike latency within each window."""
    spw    = max(1, int(1.0 / dt))
    n_tr, n_smp = data.shape
    n_bins = n_smp // spw
    trimmed = data[:, :n_bins * spw]
    binned  = trimmed.reshape(n_tr, n_bins, spw)
    amp     = np.max(np.abs(binned), axis=2)
    mask    = amp > thr
    delay   = (np.clip(1 - amp, 0, 1) * (spw - 1)).astype(int)
    starts  = np.arange(n_bins) * spw
    idx     = np.clip(starts[np.newaxis, :] + delay, 0, n_smp - 1)
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_idx  = np.repeat(np.arange(n_tr)[:, np.newaxis], n_bins, axis=1)
    out[tr_idx[mask], idx[mask]] = 1.0
    return out


def enc_ttfs(data, dt, thr):
    """
    B3: Time-To-First-Spike (TTFS) — Petro et al. TNNLS 2020.
    HIGH amplitude → EARLY spike (small latency). ONE spike per window.
    Vectorised: identical structure to Phase but uses max amplitude.
    """
    spw    = max(1, int(1.0 / dt))
    n_tr, n_smp = data.shape
    n_bins = n_smp // spw
    trimmed = data[:, :n_bins * spw]
    binned  = trimmed.reshape(n_tr, n_bins, spw)
    amp     = np.max(np.abs(binned), axis=2)            # peak amplitude per window
    mask    = amp > thr
    # TTFS: delay = (1 - amp) × (spw-1)  →  high amp fires first
    delay   = (np.clip(1.0 - amp, 0.0, 1.0) * (spw - 1)).astype(int)
    starts  = np.arange(n_bins) * spw
    idx     = np.clip(starts[np.newaxis, :] + delay, 0, n_smp - 1)
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_idx  = np.repeat(np.arange(n_tr)[:, np.newaxis], n_bins, axis=1)
    out[tr_idx[mask], idx[mask]] = 1.0
    return out


# ─────────────────────────────────────────────────────────────────────────────
# PROPOSED ENCODERS
# ─────────────────────────────────────────────────────────────────────────────

def enc_delta_mod(data, dt, thr, step_size):
    """P1: Delta Modulation — DAS-tuned step_size for seismic dynamics."""
    min_interval = max(1, int(2.0 / dt))
    n_tr, n_smp  = data.shape
    tr_range     = data.max(axis=1) - data.min(axis=1)
    delta_step   = np.maximum(step_size * tr_range, 1e-6)
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    for t in range(n_tr):
        tr = data[t]; ds = delta_step[t]
        ref = tr[0]; last = -min_interval
        for i in range(1, n_smp):
            diff = tr[i] - ref
            if abs(diff) >= ds:
                n_steps = int(abs(diff) / ds)
                ref += np.sign(diff) * n_steps * ds
                if diff > 0 and tr[i] > thr and i - last >= min_interval:
                    out[t, i] = 1.0; last = i
    return out


def enc_pde(data, dt, kappa_phi, amp_kappa, min_delta_phi=0.10):
    """P2: Phase-Delta Encoding — analytic signal phase + amplitude gate."""
    n_tr, n_smp = data.shape
    smooth_k    = max(3, int(5.0 / dt))
    kernel      = np.ones(smooth_k) / smooth_k
    X = np.fft.fft(data, axis=1)
    Z = np.zeros_like(X)
    Z[:, 0] = X[:, 0]
    if n_smp % 2 == 0:
        Z[:, 1:n_smp//2]  = 2.0 * X[:, 1:n_smp//2]
        Z[:, n_smp//2]    = X[:, n_smp//2]
    else:
        Z[:, 1:(n_smp+1)//2] = 2.0 * X[:, 1:(n_smp+1)//2]
    z = np.fft.ifft(Z, axis=1)
    env_raw   = np.abs(z)
    env       = np.apply_along_axis(
        lambda r: np.convolve(r, kernel, mode='same'), 1, env_raw)
    p25       = np.percentile(env, 25, axis=1, keepdims=True)
    med_env   = np.median(env, axis=1, keepdims=True)
    mad_env   = np.median(np.abs(env - med_env), axis=1, keepdims=True)
    mu_amp    = p25 + amp_kappa * 1.4826 * mad_env
    delta_phi = np.angle(z[:, 1:] * np.conj(z[:, :-1]))
    abs_dphi  = np.abs(delta_phi)
    med_dphi  = np.median(abs_dphi, axis=1, keepdims=True)
    mad_dphi  = np.median(np.abs(abs_dphi - med_dphi), axis=1, keepdims=True)
    thr_dphi  = np.maximum(med_dphi + kappa_phi * 1.4826 * mad_dphi,
                           min_delta_phi)
    amp_ok    = env[:, 1:] >= mu_amp
    phase_ok  = abs_dphi > thr_dphi
    out       = np.zeros((n_tr, n_smp), dtype=np.float32)
    out[:, 1:] = (amp_ok & phase_ok).astype(np.float32)
    return out


def enc_atde(data, dt, alpha, tau_ms, beta=0.01, W=50):
    """P3: ATDE — EMA noise variance → adaptive threshold θ(t) = α·σ̂ + β."""
    n_tr, n_smp = data.shape
    alpha_ema   = 1.0 - np.exp(-dt / tau_ms) if tau_ms > 0 else 0.1
    diffs       = np.diff(data, axis=1)
    sq          = diffs ** 2
    sigma_sq    = np.zeros_like(sq)
    sigma_sq[:, 0] = sq[:, 0]
    for n in range(1, n_smp - 1):
        sigma_sq[:, n] = alpha_ema * sq[:, n] + (1 - alpha_ema) * sigma_sq[:, n-1]
    sigma_hat = np.sqrt(np.maximum(sigma_sq, 1e-18))
    theta     = alpha * np.maximum(sigma_hat, 1e-9) + beta
    fire      = np.abs(diffs) > theta
    fire[:, :W] = False
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    out[:, 1:] = fire.astype(np.float32)
    return out


def enc_maste(data, dt, lags, alpha, tau_ms=50.0, beta=0.01, W=50, min_votes=2):
    """P4: MASTE — multi-lag ATDE with majority vote for wavefront detection."""
    n_tr, n_smp = data.shape
    votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_lag    = data[:, lag:] - data[:, :-lag]
        sq          = diff_lag ** 2
        alpha_ema   = 1.0 - np.exp(-dt / tau_ms) if tau_ms > 0 else 0.1
        sigma_sq    = np.zeros_like(sq)
        sigma_sq[:, 0] = sq[:, 0]
        for n in range(1, sq.shape[1]):
            sigma_sq[:, n] = alpha_ema * sq[:, n] + (1 - alpha_ema) * sigma_sq[:, n-1]
        sigma_hat = np.sqrt(np.maximum(sigma_sq, 1e-18))
        theta     = alpha * np.maximum(sigma_hat, 1e-9) + beta
        fire      = (np.abs(diff_lag) > theta).astype(np.int32)
        fire[:, :W] = 0
        layer = np.zeros((n_tr, n_smp), dtype=np.int32)
        layer[:, lag:] = fire
        votes += layer
    return (votes >= min_votes).astype(np.float32)


def enc_stmw(data, dt, thr_t, thr_s, wt_ms, ws):
    """
    P5: Spatio-Temporal Moving-Window (ST-MW).
    Extends Petro et al. TNNLS 2020 MW from 1D → 2D joint condition.
    Temporal condition vectorised; spatial condition requires channel loop.
    """
    n_tr, n_smp = data.shape
    spw_t = max(1, int(wt_ms / dt))
    spw_s = max(2, ws)
    n_bins = n_smp // spw_t
    half_s = spw_s // 2

    # ── Temporal condition (vectorised) ──────────────────────────────────────
    trimmed  = data[:, :n_bins * spw_t]
    binned   = trimmed.reshape(n_tr, n_bins, spw_t)
    abs_b    = np.abs(binned)
    da_t     = abs_b.max(axis=2) - abs_b.min(axis=2)    # (n_tr, n_bins)
    peak_off = np.argmax(abs_b, axis=2)                  # (n_tr, n_bins)
    starts   = np.arange(n_bins) * spw_t
    spike_t  = np.clip(starts[np.newaxis, :] + peak_off, 0, n_smp - 1)
    t_mids   = starts + spw_t // 2                       # (n_bins,)

    # ── Spatial condition (channel loop required for window) ─────────────────
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    for i in range(n_tr):
        ch_s = max(0, i - half_s)
        ch_e = min(n_tr, i + half_s + 1)
        for b in range(n_bins):
            if da_t[i, b] <= thr_t:
                continue
            t_m  = int(t_mids[b])
            da_s = np.max(np.abs(data[ch_s:ch_e, t_m])) \
                 - np.min(np.abs(data[ch_s:ch_e, t_m]))
            if da_s <= thr_s:
                continue
            out[i, spike_t[i, b]] = 1.0
    return out


def enc_icde(data, dt, K, alpha, encode_window_ms,
              v_min=1000.0, v_max=6000.0, beta=0.01, W=50):
    """
    P6: Inter-Channel Delay Encoding (ICDE).
    Extends DTE (arXiv:2311.14447) to DAS channel pairs.
    Onset detection fully vectorised via ATDE; channel-pair loop over traces.
    """
    n_tr, n_smp = data.shape
    dt_s      = dt / 1000.0
    encode_w  = max(1, int(encode_window_ms / dt))

    # ── Vectorised onset detection using ATDE ────────────────────────────────
    alpha_ema = 1.0 - np.exp(-dt / 35.0)   # fixed EMA constant for onset
    diffs     = np.diff(data, axis=1)
    sq        = diffs ** 2
    sigma_sq  = np.zeros_like(sq)
    sigma_sq[:, 0] = sq[:, 0]
    for n in range(1, n_smp - 1):
        sigma_sq[:, n] = alpha_ema * sq[:, n] + (1 - alpha_ema) * sigma_sq[:, n-1]
    sigma_hat = np.sqrt(np.maximum(sigma_sq, 1e-18))
    theta     = alpha * np.maximum(sigma_hat, 1e-9) + beta
    fire      = np.abs(diffs) > theta
    fire[:, :W] = False

    # First onset sample per trace (-1 = no onset)
    onset_times = np.full(n_tr, -1, dtype=int)
    has_onset   = fire.any(axis=1)
    for i in np.where(has_onset)[0]:
        onset_times[i] = int(np.argmax(fire[i])) + 1

    # ── Channel-pair loop ────────────────────────────────────────────────────
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    for i in range(n_tr - K):
        if onset_times[i] < 0 or onset_times[i + K] < 0:
            continue
        t_i  = onset_times[i]
        t_iK = onset_times[i + K]
        if t_iK == t_i:
            continue
        delta_t_s  = abs(t_iK - t_i) * dt_s
        v_apparent = (K * TRACE_SPACING_M) / delta_t_s
        if not (v_min <= v_apparent <= v_max):
            continue
        v_norm  = np.clip((v_apparent - v_min) / (v_max - v_min), 0.0, 1.0)
        offset  = int((1.0 - v_norm) * (encode_w - 1))
        spike_t = t_i + offset
        if 0 <= spike_t < n_smp:
            out[i, spike_t] = 1.0
    return out


def enc_sfbe(data, dt, alpha, tau_ms, bands=None):
    """
    P7: Sub-band Frequency Band Encoding (SFBE).
    Extends FEEL-SNN FE (NeurIPS 2024) to DAS seismic sub-bands.
    Per-band filtering and ATDE thresholding vectorised across traces.
    Time-multiplexing maps each band's spikes to a dedicated output segment.
    """
    if bands is None:
        bands = SFBE_BANDS
    fs_hz   = 1000.0 / dt
    n_tr, n_smp = data.shape
    n_bands = len(bands)
    seg_len = n_smp // n_bands
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    W       = 50
    beta    = 0.01
    alpha_ema = 1.0 - np.exp(-dt / tau_ms) if tau_ms > 0 else 0.1

    for b_idx, (lo, hi) in enumerate(bands):
        nyq   = 0.5 * fs_hz
        blow  = np.clip(lo / nyq, 1e-6, 0.99)
        bhigh = np.clip(hi / nyq, 1e-6, 0.99)
        b_c, a_c = butter(4, [blow, bhigh], btype='band')

        # Filter all traces for this sub-band
        filt_b = np.zeros_like(data)
        for t in range(n_tr):
            filt_b[t] = filtfilt(b_c, a_c, data[t])

        # Hilbert envelope — vectorised
        env_b = np.abs(hilbert(filt_b, axis=1))
        mx    = np.where(env_b.max(axis=1, keepdims=True) > 0,
                         env_b.max(axis=1, keepdims=True), 1.0)
        env_b = env_b / mx

        # ATDE on envelope — vectorised
        diffs    = np.diff(env_b, axis=1)
        sq       = diffs ** 2
        sigma_sq = np.zeros_like(sq)
        sigma_sq[:, 0] = sq[:, 0]
        for n in range(1, n_smp - 1):
            sigma_sq[:, n] = (alpha_ema * sq[:, n]
                              + (1 - alpha_ema) * sigma_sq[:, n-1])
        sigma_hat = np.sqrt(np.maximum(sigma_sq, 1e-18))
        theta     = alpha * np.maximum(sigma_hat, 1e-9) + beta
        fire      = np.abs(diffs) > theta
        fire[:, :W] = False      # warm-up period

        # Time-multiplex: map fire position n → output segment position
        # fire indices are diff positions (0..n_smp-2), actual time = n+1
        seg_start = b_idx * seg_len
        seg_end   = seg_start + seg_len if b_idx < n_bands - 1 else n_smp

        tr_idx, fire_n = np.where(fire)
        fire_t  = fire_n + 1                                  # actual time
        seg_pos = seg_start + (fire_t * seg_len // n_smp)     # target position
        valid   = (seg_pos >= seg_start) & (seg_pos < seg_end) & (seg_pos < n_smp)
        out[tr_idx[valid], seg_pos[valid]] = 1.0

    return out


# =============================================================================
# SWEEP RUNNER
# =============================================================================

def run_sweep_two_class(name, param_combos, encode_fn,
                        event_data, noise_data, key='norm_data'):
    """
    Separate event and noise passes.
    SNR computed on event files only (noise has no expected event window).
    Sparsity averaged across both classes.
    """
    results  = []
    n_combos = len(param_combos)

    for ci, params in enumerate(param_combos):
        ev_spk, ev_dt = [], []
        for rec in event_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                ev_spk.append(spk); ev_dt.append(rec['dt_ms'])
            except Exception:
                continue

        no_spk = []
        for rec in noise_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                no_spk.append(spk)
            except Exception:
                continue

        if not ev_spk:
            continue

        ev_sp, ev_snr, ev_prec = batch_metrics(ev_spk, ev_dt)
        no_sp   = float(np.mean([s.mean() for s in no_spk])) if no_spk else ev_sp
        sp_avg  = (ev_sp + no_sp) / 2
        in_target = TARGET_SP_LO <= sp_avg <= TARGET_SP_HI

        results.append({**params,
                        'sparsity_event': ev_sp,
                        'sparsity_noise': no_sp,
                        'sparsity_avg':   sp_avg,
                        'snr':            ev_snr,
                        'precision':      ev_prec,
                        'in_target':      in_target})

        if (ci+1) % 5 == 0 or (ci+1) == n_combos:
            print(f"    [{ci+1:>3d}/{n_combos}]  "
                  f"sp_ev={ev_sp*100:.3f}%  sp_no={no_sp*100:.3f}%  "
                  f"sp_avg={sp_avg*100:.3f}%  snr={ev_snr:.2f}×  "
                  f"{'✓' if in_target else '✗'}",
                  flush=True)

    return results


# =============================================================================
# REPORTING
# =============================================================================

def report(name, results, param_names, all_rows):
    if not results:
        print(f"\n  {name}: no results"); return None

    target = [r for r in results if r['in_target']]
    target.sort(key=lambda x: x['snr'], reverse=True)
    sp_key = 'sparsity_avg' if 'sparsity_avg' in results[0] else 'sparsity'

    print(f"\n  {'─'*80}")
    print(f"  {name}: {len(target)}/{len(results)} configs in target "
          f"({TARGET_SP_LO*100:.1f}%–{TARGET_SP_HI*100:.1f}%)")
    print(f"  {'─'*80}")

    show = target[:8] if target else sorted(
        results, key=lambda x: abs(x[sp_key] - 0.007))[:8]

    for r in show:
        vals = '  '.join(
            f'{r[p]:>10.4f}' if isinstance(r[p], float)
            else f'{str(r[p]):>10}' for p in param_names)
        tag = '★' if r['in_target'] else ' '
        print(f"  {vals}  sp_avg={r[sp_key]*100:.3f}%  "
              f"snr={r['snr']:.2f}×  "
              f"prec={r['precision']:.3f}  {tag}")

    best = target[0] if target else None
    if best:
        skip = {'sparsity', 'sparsity_event', 'sparsity_noise',
                'sparsity_avg', 'snr', 'precision', 'in_target'}
        rec  = ', '.join(f'{k}={v}' for k, v in best.items() if k not in skip)
        print(f"\n  → BEST: {rec}")
        print(f"    sp_avg={best[sp_key]*100:.3f}%  "
              f"snr={best['snr']:.2f}×  "
              f"precision={best['precision']:.3f}")
    else:
        print(f"  → No config in target range. Closest shown above.")

    for r in results:
        all_rows.append({'encoder': name, **r})

    return best


# =============================================================================
# MAIN — WIDE PARAMETER SWEEPS
# =============================================================================

def run_full_sweep():
    print("=" * 80)
    print("MULTI-FILE VECTORISED PARAMETER SWEEP — 10 ENCODERS (WIDE RANGES)")
    print(f"  Event dir:  {EVENT_DIR}")
    print(f"  Noise dir:  {NOISE_DIR}")
    print(f"  Samples:    {N_SAMPLES} event + {N_SAMPLES} noise = {2*N_SAMPLES} files")
    print(f"  Target sp:  {TARGET_SP_LO*100:.1f}% – {TARGET_SP_HI*100:.1f}%")
    print("=" * 80)

    print("\nSampling files...")
    event_files = sample_files(EVENT_DIR, N_SAMPLES, RANDOM_SEED)
    noise_files = sample_files(NOISE_DIR, N_SAMPLES, RANDOM_SEED + 1)
    print(f"  Event: {len(event_files)} | Noise: {len(noise_files)}")

    t0 = time.time()
    event_data, noise_data = load_dataset(event_files, noise_files)
    print(f"  Preprocessing: {(time.time()-t0)/60:.1f} min")

    if not event_data or not noise_data:
        print("ERROR: Could not load files. Check paths."); return

    all_rows    = []
    best_params = {}
    t_total     = time.time()


    # ── [B1] Rate ────────────────────────────────────────────────────────────
    # thr: 0.40 → 0.95  (12 values)
    print("\n [1/10] Rate  (B1 — baseline)...")
    t0 = time.time()
    combos = [{'thr': t}
              for t in np.round(np.arange(0.40, 0.96, 0.05), 4)]
    print(f"  Grid: thr {[c['thr'] for c in combos]} → {len(combos)} combos")
    res = run_sweep_two_class('Rate', combos, enc_rate,
                               event_data, noise_data, 'norm_data')
    best_params['Rate'] = report('Rate', res, ['thr'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [B2] Phase ───────────────────────────────────────────────────────────
    # thr: 0.40 → 0.95  (12 values)
    print("\n [2/10] Phase  (B2 — baseline)...")
    t0 = time.time()
    combos = [{'thr': t}
              for t in np.round(np.arange(0.40, 0.96, 0.05), 4)]
    print(f"  Grid: thr {[c['thr'] for c in combos]} → {len(combos)} combos")
    res = run_sweep_two_class('Phase', combos, enc_phase,
                               event_data, noise_data, 'norm_data')
    best_params['Phase'] = report('Phase', res, ['thr'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [B3] TTFS ────────────────────────────────────────────────────────────
    # thr: 0.40 → 0.95  (12 values)
    print("\n [3/10] TTFS  (B3 — baseline, Petro et al. TNNLS 2020)...")
    t0 = time.time()
    combos = [{'thr': t}
              for t in np.round(np.arange(0.40, 0.96, 0.05), 4)]
    print(f"  Grid: thr {[c['thr'] for c in combos]} → {len(combos)} combos")
    res = run_sweep_two_class('TTFS', combos, enc_ttfs,
                               event_data, noise_data, 'norm_data')
    best_params['TTFS'] = report('TTFS', res, ['thr'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P1] Delta-Mod ────────────────────────────────────────────────────────
    # thr:       0.40 → 0.95 (12 values)
    # step_size: 0.02 → 0.30 (11 values)
    # Total: 132 combos
    print("\n [4/10] Delta-Mod  (P1 — proposed)...")
    t0 = time.time()
    thr_vals  = np.round(np.arange(0.40, 0.96, 0.05), 4)
    step_vals = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]
    combos    = [{'thr': t, 'step_size': ss}
                 for t in thr_vals for ss in step_vals]
    print(f"  Grid: thr(12) × step_size(11) = {len(combos)} combos")
    res = run_sweep_two_class('Delta-Mod', combos, enc_delta_mod,
                               event_data, noise_data, 'norm_data')
    best_params['Delta-Mod'] = report('Delta-Mod', res,
                                       ['thr', 'step_size'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P2] PDE ──────────────────────────────────────────────────────────────
    # kappa_phi: 0.5 → 4.0  (10 values)
    # amp_kappa: 0.3 → 3.5  (10 values)
    # Total: 100 combos
    print("\n [5/10] PDE  (P2 — proposed)...")
    t0 = time.time()
    kphi_vals = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
    kamp_vals = [0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 3.5]
    combos    = [{'kappa_phi': kp, 'amp_kappa': ka}
                 for kp in kphi_vals for ka in kamp_vals]
    print(f"  Grid: kappa_phi(10) × amp_kappa(10) = {len(combos)} combos")
    res = run_sweep_two_class('PDE', combos, enc_pde,
                               event_data, noise_data, 'filtered')
    best_params['PDE'] = report('PDE', res,
                                 ['kappa_phi', 'amp_kappa'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P3] ATDE ─────────────────────────────────────────────────────────────
    # alpha:  0.5 → 5.0    (10 values)
    # tau_ms: 5.0 → 200.0  (10 values)
    # Total: 100 combos
    print("\n [6/10] ATDE  (P3 — proposed)...")
    t0 = time.time()
    alpha_vals = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
    tau_vals   = [5.0, 10.0, 20.0, 35.0, 50.0, 75.0, 100.0, 130.0, 165.0, 200.0]
    combos     = [{'alpha': a, 'tau_ms': t}
                  for a in alpha_vals for t in tau_vals]
    print(f"  Grid: alpha(10) × tau_ms(10) = {len(combos)} combos")
    res = run_sweep_two_class('ATDE', combos, enc_atde,
                               event_data, noise_data, 'norm_data')
    best_params['ATDE'] = report('ATDE', res,
                                  ['alpha', 'tau_ms'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P4] MASTE ────────────────────────────────────────────────────────────
    # lags:  8 lag combinations
    # alpha: 1.0 → 4.5  (8 values)
    # Total: 64 combos
    print("\n [7/10] MASTE  (P4 — proposed)...")
    t0 = time.time()
    lag_sets = [
        [1, 2, 4],       # 0.5, 1.0, 2.0 ms — short-scale
        [1, 3, 8],       # 0.5, 1.5, 4.0 ms — original best
        [1, 4, 12],      # 0.5, 2.0, 6.0 ms — medium-scale
        [1, 5, 15],      # 0.5, 2.5, 7.5 ms — longer-scale
        [2, 4, 8],       # 1.0, 2.0, 4.0 ms — even spacing
        [2, 5, 10],      # 1.0, 2.5, 5.0 ms — medium-even
        [3, 6, 12],      # 1.5, 3.0, 6.0 ms — coarse
        [1, 2, 5, 10],   # 0.5, 1.0, 2.5, 5.0 ms — 4-lag set
    ]
    alpha_vals = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]
    combos     = [{'lags': lags, 'alpha': a}
                  for lags in lag_sets for a in alpha_vals]
    print(f"  Grid: lags(8) × alpha(8) = {len(combos)} combos")
    res = run_sweep_two_class('MASTE', combos, enc_maste,
                               event_data, noise_data, 'norm_data')
    best_params['MASTE'] = report('MASTE', res,
                                   ['lags', 'alpha'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P5] ST-MW ────────────────────────────────────────────────────────────
    # thr_t: 0.10 → 0.58  (8 values)
    # thr_s: 0.05 → 0.45  (6 values)
    # wt_ms: 1.0, 2.0, 4.0  (3 values)
    # ws:    4, 8, 16  (3 values)
    # Total: 8 × 6 × 3 × 3 = 432 combos
    print("\n [8/10] ST-MW  (P5 — proposed, Petro TNNLS 2020 → 2D)...")
    t0 = time.time()
    thr_t_vals = np.round(np.arange(0.10, 0.66, 0.08), 3)
    thr_s_vals = np.round(np.arange(0.05, 0.46, 0.08), 3)
    wt_vals    = [1.0, 2.0, 4.0]
    ws_vals    = [4, 8, 16]
    combos     = [{'thr_t': tt, 'thr_s': ts, 'wt_ms': wt, 'ws': ws}
                  for tt in thr_t_vals
                  for ts in thr_s_vals
                  for wt in wt_vals
                  for ws in ws_vals]
    print(f"  Grid: thr_t(8) × thr_s(6) × wt_ms(3) × ws(3) = {len(combos)} combos")
    res = run_sweep_two_class('ST-MW', combos, enc_stmw,
                               event_data, noise_data, 'norm_data')
    best_params['ST-MW'] = report('ST-MW', res,
                                   ['thr_t', 'thr_s', 'wt_ms', 'ws'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P6] ICDE ─────────────────────────────────────────────────────────────
    # K:                 4, 6, 8, 12, 16  (5 values)
    # alpha:             1.0 → 4.0  (7 values)
    # encode_window_ms:  5, 10, 15, 20, 30  (5 values)
    # Total: 5 × 7 × 5 = 175 combos
    print("\n [9/10] ICDE  (P6 — proposed, DTE arXiv:2311.14447 → DAS)...")
    t0 = time.time()
    K_vals  = [4, 6, 8, 12, 16]
    a_vals  = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
    ew_vals = [5.0, 10.0, 15.0, 20.0, 30.0]
    combos  = [{'K': K, 'alpha': a, 'encode_window_ms': ew}
               for K in K_vals for a in a_vals for ew in ew_vals]
    print(f"  Grid: K(5) × alpha(7) × encode_window_ms(5) = {len(combos)} combos")
    print(f"  (v_min=1000 m/s, v_max=6000 m/s — seismic physics, not swept)")
    res = run_sweep_two_class('ICDE', combos, enc_icde,
                               event_data, noise_data, 'norm_data')
    best_params['ICDE'] = report('ICDE', res,
                                  ['K', 'alpha', 'encode_window_ms'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── [P7] SFBE ─────────────────────────────────────────────────────────────
    # alpha:  0.5 → 5.0    (10 values)
    # tau_ms: 5.0 → 200.0  (10 values)
    # Total: 10 × 10 = 100 combos
    # Sub-bands fixed: [(50,100),(100,150),(150,200),(200,250)]
    print("\n [10/10] SFBE  (P7 — proposed, FEEL-SNN NeurIPS 2024 → DAS)...")
    t0 = time.time()
    alpha_vals = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
    tau_vals   = [5.0, 10.0, 20.0, 35.0, 50.0, 75.0, 100.0, 130.0, 165.0, 200.0]
    combos     = [{'alpha': a, 'tau_ms': t}
                  for a in alpha_vals for t in tau_vals]
    print(f"  Grid: alpha(10) × tau_ms(10) = {len(combos)} combos")
    print(f"  (sub-bands {SFBE_BANDS} — seismic-physics fixed, not swept)")
    res = run_sweep_two_class('SFBE', combos, enc_sfbe,
                               event_data, noise_data, 'norm_data')
    best_params['SFBE'] = report('SFBE', res,
                                  ['alpha', 'tau_ms'], all_rows)
    print(f"  Time: {(time.time()-t0)/60:.1f} min")


    # ── Save results ─────────────────────────────────────────────────────────
    csv_path = os.path.join(RESULTS_DIR, 'sweep_results_multifile.csv')
    pd.DataFrame(all_rows).to_csv(csv_path, index=False)
    print(f"\n Full results → {csv_path}")

    json_path = os.path.join(RESULTS_DIR, 'best_params_multifile.json')
    skip      = {'sparsity', 'sparsity_event', 'sparsity_noise',
                 'sparsity_avg', 'snr', 'precision', 'in_target'}
    serialisable = {}
    for enc, best in best_params.items():
        if best is None:
            serialisable[enc] = None; continue
        serialisable[enc] = {
            k: (list(v) if isinstance(v, (list, np.ndarray)) else
                float(v) if isinstance(v, (np.floating, float)) else v)
            for k, v in best.items() if k not in skip}
    with open(json_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f" Best params → {json_path}")

    # ── Final summary ─────────────────────────────────────────────────────────
    encoder_types = {
        'Rate':      'Baseline', 'Phase':  'Baseline', 'TTFS':  'Baseline',
        'Delta-Mod': 'Proposed', 'PDE':    'Proposed', 'ATDE':  'Proposed',
        'MASTE':     'Proposed', 'ST-MW':  'Proposed', 'ICDE':  'Proposed',
        'SFBE':      'Proposed',
    }
    print(f"\n\n{'='*100}")
    print(f"FINAL RECOMMENDED PARAMETERS — {N_SAMPLES} event + {N_SAMPLES} noise files")
    print(f"{'='*100}")
    print(f"  {'Encoder':>12}  {'Type':>9}  {'sp_ev%':>7}  {'sp_no%':>7}  "
          f"{'sp_avg%':>8}  {'SNR':>6}  {'Prec':>6}  Parameters")
    print(f"  {'─'*12}  {'─'*9}  {'─'*7}  {'─'*7}  {'─'*8}  {'─'*6}  {'─'*6}  {'─'*35}")

    for name, best in best_params.items():
        etype = encoder_types.get(name, '')
        if best is None:
            print(f"  {name:>12}  {etype:>9}  NO CONFIG IN TARGET RANGE"); continue
        skip_s = {'sparsity', 'sparsity_event', 'sparsity_noise',
                  'sparsity_avg', 'snr', 'precision', 'in_target'}
        params_str = ', '.join(
            f'{k}={v}' for k, v in best.items() if k not in skip_s)
        sp_ev = best.get('sparsity_event', best.get('sparsity', 0))
        sp_no = best.get('sparsity_noise', best.get('sparsity', 0))
        sp_av = best.get('sparsity_avg',   best.get('sparsity', 0))
        print(f"  {name:>12}  {etype:>9}  {sp_ev*100:>7.3f}  {sp_no*100:>7.3f}  "
              f"{sp_av*100:>8.3f}  {best['snr']:>6.2f}  "
              f"{best['precision']:>6.3f}  {params_str}")

    elapsed = time.time() - t_total
    print(f"\n  Total time: {elapsed/60:.1f} min  "
          f"({len(event_data)} event + {len(noise_data)} noise files)")
    print(f"""
  ─────────────────────────────────────────────────────────────────────────────
  SWEEP GRID SUMMARY (wide ranges — same as single-file sweep)
  ─────────────────────────────────────────────────────────────────────────────
  [B1] Rate       thr: 0.40–0.95 (step 0.05)                      12 values
  [B2] Phase      thr: 0.40–0.95 (step 0.05)                      12 values
  [B3] TTFS       thr: 0.40–0.95 (step 0.05)                      12 values
  [P1] Delta-Mod  thr: 0.40–0.95 × step_size: 0.02–0.30      132 combos
  [P2] PDE        kappa_phi: 0.5–4.0 × amp_kappa: 0.3–3.5    100 combos
  [P3] ATDE       alpha: 0.5–5.0 × tau_ms: 5–200 ms           100 combos
  [P4] MASTE      8 lag-sets × alpha: 1.0–4.5                   64 combos
  [P5] ST-MW      thr_t(8) × thr_s(6) × wt_ms(3) × ws(3)    432 combos
  [P6] ICDE       K(5) × alpha(7) × encode_window_ms(5)       175 combos
  [P7] SFBE       alpha: 0.5–5.0 × tau_ms: 5–200 ms           100 combos
                                               Total:        1,139 combos
  ─────────────────────────────────────────────────────────────────────────────
  Copy best_params_multifile.json → 3__Preprocessing__10k_files_.ipynb
  Use sp_avg (event+noise average) as the primary selection criterion.
  For final paper params, prefer configs where sp_ev > sp_no (event fires more
  than noise — good discriminability for the SNN classifier).
""")
    return best_params


if __name__ == "__main__":
    run_full_sweep()

In [ ]:
"""
PATCH SWEEP — FIXED ENCODERS ONLY (5 of 10)
============================================
Runs ONLY the 5 encoders that were fixed/replaced in sweep v2.
Uses v1 confirmed best params for the other 5 (no re-run needed).

FIXED ENCODERS TO RUN:
  [B3] TTFS      — FIX 1: first threshold-crossing (was identical to Phase)
  [P1] Delta-Mod — FIX 4: vectorised numpy        (was 1484 min Python loop)
  [P5] ST-MW     — FIX 5: scipy spatial filter    (was 2958 min channel loop)
  [P6] CSS       — NEW: replaces failed ICDE       (was 0/175 in target)
  [P7] SFBE      — FIX 3: STA/LTA energy ratio    (was SNR=1.26×)

V1 CONFIRMED BEST (loaded from JSON, not re-run):
  Rate:  thr=0.85           → SNR=3.09×  sp=0.461%  prec=0.253
  Phase: thr=0.85           → SNR=2.71×  sp=0.507%  prec=0.249
  PDE:   kappa_phi=0.5,
         amp_kappa=3.0      → SNR=2.17×  sp=0.563%  prec=0.268
  ATDE:  alpha=3.0,
         tau_ms=165.0       → SNR=2.22×  sp=0.349%  prec=0.208
  MASTE: lags=[1,3,8],
         alpha=2.5          → SNR=2.52×  sp=0.311%  prec=0.264

Estimated runtime:
  TTFS:      ~3 min   (12 combos,  vectorised)
  Delta-Mod: ~5 min   (132 combos, vectorised FIX)
  ST-MW:     ~15 min  (960 combos, scipy FIX)
  CSS:       ~8 min   (60 combos,  new)
  SFBE:      ~5 min   (48 combos,  STA/LTA FIX)
  TOTAL:     ~36 min

Output:
  best_params_patch.json  — all 10 encoders (v1 confirmed + patch results)
  sweep_results_patch.csv — patch sweep rows only
"""

import os, json, time, random
import segyio
import numpy as np
import pandas as pd
from scipy.signal  import butter, filtfilt, hilbert
from scipy.ndimage import maximum_filter1d, minimum_filter1d

# =============================================================================
# CONFIGURATION
# =============================================================================
EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
RESULTS_DIR = config.SWEEP_RESULTS_DIR

N_SAMPLES   = 400      # keep same as v1 for fair comparison
RANDOM_SEED = 42       # MUST match v1 — same files sampled

FORCE_DT_MS     = 0.5
FILTER_LOW_HZ   = 50
FILTER_HIGH_HZ  = 250
WINDOW_START_MS = 0
WINDOW_END_MS   = 1000
EVENT_START_MS  = 380.0
EVENT_END_MS    = 560.0

TARGET_SP_LO = 0.003
TARGET_SP_HI = 0.015

TRACE_SPACING_M = 4.0
SFBE_BANDS      = [(50, 100), (100, 150), (150, 200), (200, 250)]

os.makedirs(RESULTS_DIR, exist_ok=True)

# =============================================================================
# V1 CONFIRMED BEST PARAMS (hard-coded from sweep v1 results)
# These are NOT re-run — taken directly from your sweep output.
# =============================================================================
V1_CONFIRMED = {
    'Rate':  {'thr': 0.85,
              'sparsity_event': 0.00409, 'sparsity_noise': 0.00514,
              'sparsity_avg': 0.00461, 'snr': 3.09, 'precision': 0.253},
    'Phase': {'thr': 0.85,
              'sparsity_event': 0.00444, 'sparsity_noise': 0.00570,
              'sparsity_avg': 0.00507, 'snr': 2.71, 'precision': 0.249},
    'PDE':   {'kappa_phi': 0.5, 'amp_kappa': 3.0,
              'sparsity_event': 0.00672, 'sparsity_noise': 0.00455,
              'sparsity_avg': 0.00563, 'snr': 2.17, 'precision': 0.268},
    'ATDE':  {'alpha': 3.0, 'tau_ms': 165.0,
              'sparsity_event': 0.00416, 'sparsity_noise': 0.00282,
              'sparsity_avg': 0.00349, 'snr': 2.22, 'precision': 0.208},
    'MASTE': {'lags': [1, 3, 8], 'alpha': 2.5,
              'sparsity_event': 0.00366, 'sparsity_noise': 0.00256,
              'sparsity_avg': 0.00311, 'snr': 2.52, 'precision': 0.264},
}


# =============================================================================
# FILE SAMPLING + LOAD  (identical to v1)
# =============================================================================
def sample_files(directory, n, seed):
    all_files = sorted([
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(('.sgy', '.segy'))
    ])
    if len(all_files) < n:
        print(f"   Only {len(all_files)} files — using all.")
        return all_files
    rng = random.Random(seed)
    return sorted(rng.sample(all_files, n))


def load_and_preprocess(filepath):
    try:
        with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))],
                            dtype=np.float64)
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                if not (0.001 <= dt_ms <= 10.0):
                    dt_ms = FORCE_DT_MS
            except Exception:
                dt_ms = FORCE_DT_MS
    except Exception as ex:
        print(f"     {os.path.basename(filepath)}: {ex}")
        return None

    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ/nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ/nyq, 1e-6, 0.99)], btype='band')
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        filtered[i] = filtfilt(b, a, data[i])

    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(filtered.shape[1], int(WINDOW_END_MS / dt_ms))
    filtered = filtered[:, s:e]

    tmp  = filtered - filtered.mean(axis=1, keepdims=True)
    mx   = np.where(np.max(np.abs(tmp), axis=1, keepdims=True) > 0,
                    np.max(np.abs(tmp), axis=1, keepdims=True), 1.0)
    norm_data = (tmp / mx + 1) / 2

    env_raw  = np.abs(hilbert(filtered, axis=1))
    mx_env   = np.where(env_raw.max(axis=1, keepdims=True) > 0,
                        env_raw.max(axis=1, keepdims=True), 1.0)
    env      = env_raw / mx_env
    tmp2     = env - env.mean(axis=1, keepdims=True)
    mx2      = np.where(np.max(np.abs(tmp2), axis=1, keepdims=True) > 0,
                        np.max(np.abs(tmp2), axis=1, keepdims=True), 1.0)
    norm_env = (tmp2 / mx2 + 1) / 2

    return {'norm_data': norm_data, 'norm_env': norm_env,
            'filtered': filtered, 'dt_ms': dt_ms}


def load_dataset(event_files, noise_files):
    print(f"\n  Preprocessing {len(event_files)} event files...")
    event_data = []
    for i, fp in enumerate(event_files):
        r = load_and_preprocess(fp)
        if r is not None:
            event_data.append(r)
        if (i+1) % 100 == 0:
            print(f"    {i+1}/{len(event_files)}")

    print(f"  Preprocessing {len(noise_files)} noise files...")
    noise_data = []
    for i, fp in enumerate(noise_files):
        r = load_and_preprocess(fp)
        if r is not None:
            noise_data.append(r)
        if (i+1) % 100 == 0:
            print(f"    {i+1}/{len(noise_files)}")

    print(f"  Loaded: {len(event_data)} event + {len(noise_data)} noise")
    return event_data, noise_data


# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(spikes, dt_ms):
    abs_spk = np.abs(spikes)           # handles CSS ternary {+1,0,-1}
    n_tr, n_smp = abs_spk.shape
    sp  = float(abs_spk.mean())
    s   = max(0, int(EVENT_START_MS / dt_ms))
    e   = min(n_smp, int(EVENT_END_MS / dt_ms))
    density = abs_spk.mean(axis=0)
    in_ev   = float(density[s:e].mean()) if e > s else 0.0
    out_smp = np.concatenate([density[:s], density[e:]])
    min_d   = 1.0 / max(n_tr * n_smp, 1)
    out_ev  = float(max(out_smp.mean(), min_d)) if len(out_smp) > 0 else min_d
    snr     = in_ev / out_ev if out_ev > 0 else 0.0
    total   = float(abs_spk.sum())
    prec    = float(abs_spk[:, s:e].sum() / total) if total > 0 else 0.0
    return sp, snr, prec


def batch_metrics(spikes_list, dt_ms_list):
    sps, snrs, precs = [], [], []
    for spk, dt in zip(spikes_list, dt_ms_list):
        sp, snr, prec = compute_metrics(spk, dt)
        sps.append(sp); snrs.append(snr); precs.append(prec)
    return float(np.mean(sps)), float(np.mean(snrs)), float(np.mean(precs))


# =============================================================================
# FIXED ENCODERS ONLY
# =============================================================================

def enc_ttfs(data, dt, thr):
    """B3 FIX1 — first threshold-crossing (not peak-amplitude delay)."""
    spw    = max(1, int(1.0 / dt))
    n_tr, n_smp = data.shape
    n_bins = n_smp // spw
    trimmed = data[:, :n_bins * spw]
    binned  = trimmed.reshape(n_tr, n_bins, spw)
    abs_b   = np.abs(binned)
    any_cross   = (abs_b > thr).any(axis=2)
    first_cross = np.argmax(abs_b > thr, axis=2)
    starts = np.arange(n_bins) * spw
    idx    = np.clip(starts[np.newaxis, :] + first_cross, 0, n_smp - 1)
    out    = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_idx = np.repeat(np.arange(n_tr)[:, np.newaxis], n_bins, axis=1)
    out[tr_idx[any_cross], idx[any_cross]] = 1.0
    return out


def enc_delta_mod(data, dt, thr, step_size):
    """P1 FIX4 — vectorised numpy (no Python sample loop)."""
    n_tr, n_smp = data.shape
    diffs    = np.diff(data, axis=1)
    tr_range = data.max(axis=1) - data.min(axis=1)
    step     = np.maximum(step_size * tr_range, 1e-6)
    pos_step = diffs >= step[:, np.newaxis]
    above_thr = data[:, 1:] > thr
    fire     = pos_step & above_thr
    min_interval = max(1, int(2.0 / dt))
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    if min_interval <= 1:
        out[:, 1:] = fire.astype(np.float32)
    else:
        for t in range(n_tr):
            spike_idx = np.where(fire[t])[0] + 1
            last = -min_interval
            for idx in spike_idx:
                if idx - last >= min_interval:
                    out[t, idx] = 1.0
                    last = idx
    return out


def enc_stmw(data, dt, thr_t, thr_s, wt_ms, ws):
    """P5 FIX5 — scipy spatial filter (no channel loop)."""
    n_tr, n_smp = data.shape
    spw_t  = max(1, int(wt_ms / dt))
    spw_s  = max(2, ws)
    n_bins = n_smp // spw_t
    trimmed  = data[:, :n_bins * spw_t]
    binned   = trimmed.reshape(n_tr, n_bins, spw_t)
    abs_b    = np.abs(binned)
    da_t     = abs_b.max(axis=2) - abs_b.min(axis=2)
    peak_off = np.argmax(abs_b, axis=2)
    starts   = np.arange(n_bins) * spw_t
    spike_t  = np.clip(starts[np.newaxis, :] + peak_off, 0, n_smp - 1)
    t_mids   = (starts + spw_t // 2).clip(0, n_smp - 1)
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=spw_s, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=spw_s, axis=0, mode='nearest')
    da_s     = (spat_max - spat_min)[:, t_mids]
    both_met = (da_t > thr_t) & (da_s > thr_s)
    out      = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_ii, b_ii = np.where(both_met)
    if len(tr_ii):
        out[tr_ii, spike_t[tr_ii, b_ii]] = 1.0
    return out


def enc_css(data, dt, v_th, momentum=2.0):
    """P6 NEW — CSS (arXiv:2408.17245). Replaces failed ICDE."""
    n_tr, n_smp = data.shape
    tr_range = data.max(axis=1) - data.min(axis=1)
    vth      = np.maximum(v_th * tr_range, 1e-6)[:, np.newaxis]
    pos_thr  =  vth / 2.0
    neg_thr  = -vth / 2.0
    membrane = np.zeros((n_tr, n_smp), dtype=np.float64)
    membrane[:, 0] = data[:, 0]
    for t in range(1, n_smp):
        membrane[:, t] = momentum * membrane[:, t-1] + data[:, t]
    pos_fire = membrane >= pos_thr
    neg_fire = membrane <= neg_thr
    out = np.zeros((n_tr, n_smp), dtype=np.float32)
    out[pos_fire] =  1.0
    out[neg_fire] = -1.0
    vth_flat = (vth * np.ones((1, n_smp))).reshape(n_tr, n_smp)
    membrane[pos_fire] -= vth_flat[pos_fire]
    membrane[neg_fire] += vth_flat[neg_fire]
    return out


def enc_sfbe(data, dt, ratio_thr, sta_ms, bands=None):
    """P7 FIX3 — STA/LTA energy ratio per sub-band. Was SNR=1.26×."""
    if bands is None:
        bands = SFBE_BANDS
    fs_hz   = 1000.0 / dt
    n_tr, n_smp = data.shape
    n_bands = len(bands)
    seg_len = n_smp // n_bands
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    sta_w   = max(1, int(sta_ms / dt))
    for b_idx, (lo, hi) in enumerate(bands):
        nyq   = 0.5 * fs_hz
        blow  = np.clip(lo / nyq, 1e-6, 0.99)
        bhigh = np.clip(hi / nyq, 1e-6, 0.99)
        b_c, a_c = butter(4, [blow, bhigh], btype='band')
        filt_b = np.zeros_like(data)
        for t in range(n_tr):
            filt_b[t] = filtfilt(b_c, a_c, data[t])
        env_sq = np.abs(hilbert(filt_b, axis=1)) ** 2
        cs     = np.cumsum(env_sq, axis=1)
        sta    = np.zeros_like(env_sq)
        sta[:, :sta_w] = cs[:, :sta_w] / np.arange(1, sta_w + 1)
        sta[:, sta_w:] = (cs[:, sta_w:] - cs[:, :-sta_w]) / sta_w
        lta    = np.maximum(cs / np.arange(1, n_smp + 1), 1e-12)
        fire   = (sta / lta) > ratio_thr
        fire[:, :sta_w] = False
        seg_start = b_idx * seg_len
        seg_end   = seg_start + seg_len if b_idx < n_bands - 1 else n_smp
        tr_ii, t_ii = np.where(fire)
        seg_pos = seg_start + (t_ii * seg_len // n_smp)
        valid   = ((seg_pos >= seg_start) & (seg_pos < seg_end)
                   & (seg_pos < n_smp))
        out[tr_ii[valid], seg_pos[valid]] = 1.0
    return out


# =============================================================================
# SWEEP RUNNER
# =============================================================================
def run_sweep_two_class(name, param_combos, encode_fn,
                        event_data, noise_data, key='norm_data'):
    results  = []
    n_combos = len(param_combos)
    for ci, params in enumerate(param_combos):
        ev_spk, ev_dt = [], []
        for rec in event_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                ev_spk.append(spk); ev_dt.append(rec['dt_ms'])
            except Exception:
                continue
        no_spk = []
        for rec in noise_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                no_spk.append(spk)
            except Exception:
                continue
        if not ev_spk:
            continue
        ev_sp, ev_snr, ev_prec = batch_metrics(ev_spk, ev_dt)
        no_sp   = float(np.mean([np.abs(s).mean() for s in no_spk])) \
                  if no_spk else ev_sp
        sp_avg  = (ev_sp + no_sp) / 2
        in_target = TARGET_SP_LO <= sp_avg <= TARGET_SP_HI
        results.append({**params,
                        'sparsity_event': ev_sp,
                        'sparsity_noise': no_sp,
                        'sparsity_avg':   sp_avg,
                        'snr':            ev_snr,
                        'precision':      ev_prec,
                        'in_target':      in_target})
        if (ci+1) % 5 == 0 or (ci+1) == n_combos:
            print(f"    [{ci+1:>3d}/{n_combos}]  "
                  f"sp_ev={ev_sp*100:.3f}%  sp_no={no_sp*100:.3f}%  "
                  f"sp_avg={sp_avg*100:.3f}%  snr={ev_snr:.2f}×  "
                  f"{'✓' if in_target else '✗'}",
                  flush=True)
    return results


def report(name, results, param_names, all_rows, fix_label=''):
    if not results:
        print(f"\n  {name}: no results"); return None
    target = [r for r in results if r['in_target']]
    target.sort(key=lambda x: x['snr'], reverse=True)
    sp_key = 'sparsity_avg'
    print(f"\n  {'─'*80}")
    print(f"  {name} {fix_label}: {len(target)}/{len(results)} in target "
          f"({TARGET_SP_LO*100:.1f}%–{TARGET_SP_HI*100:.1f}%)")
    print(f"  {'─'*80}")
    show = target[:6] if target else sorted(
        results, key=lambda x: abs(x[sp_key] - 0.007))[:6]
    for r in show:
        vals = '  '.join(
            f'{r[p]:>10.4f}' if isinstance(r[p], float)
            else f'{str(r[p]):>12}' for p in param_names)
        tag = '★' if r['in_target'] else ' '
        print(f"  {vals}  sp_avg={r[sp_key]*100:.3f}%  "
              f"snr={r['snr']:.2f}×  prec={r['precision']:.3f}  {tag}")
    best = target[0] if target else None
    if best:
        skip = {'sparsity', 'sparsity_event', 'sparsity_noise',
                'sparsity_avg', 'snr', 'precision', 'in_target'}
        rec  = ', '.join(f'{k}={v}' for k, v in best.items() if k not in skip)
        print(f"\n  → BEST: {rec}")
        print(f"    sp_avg={best[sp_key]*100:.3f}%  "
              f"snr={best['snr']:.2f}×  precision={best['precision']:.3f}")
    else:
        closest = sorted(results,
                         key=lambda x: abs(x[sp_key] - 0.007))[0]
        rec = ', '.join(f'{k}={v}' for k, v in closest.items()
                        if k not in {'sparsity','sparsity_event',
                                     'sparsity_noise','sparsity_avg',
                                     'snr','precision','in_target'})
        print(f"  → No config in target. Closest: {rec}  "
              f"sp={closest[sp_key]*100:.3f}%  snr={closest['snr']:.2f}×")
    for r in results:
        all_rows.append({'encoder': name, **r})
    return best


# =============================================================================
# MAIN
# =============================================================================
def run_patch_sweep():
    print("=" * 70)
    print("PATCH SWEEP — FIXED ENCODERS ONLY (5/10)")
    print(f"  Samples: {N_SAMPLES} event + {N_SAMPLES} noise = {2*N_SAMPLES} files")
    print(f"  Target:  {TARGET_SP_LO*100:.1f}% – {TARGET_SP_HI*100:.1f}%  sparsity")
    print("  Encoders: TTFS | Delta-Mod | ST-MW | CSS | SFBE")
    print("=" * 70)

    # Load same files as v1 (same seed)
    print("\nSampling files (same seed as v1)...")
    event_files = sample_files(EVENT_DIR, N_SAMPLES, RANDOM_SEED)
    noise_files = sample_files(NOISE_DIR, N_SAMPLES, RANDOM_SEED + 1)

    t0 = time.time()
    event_data, noise_data = load_dataset(event_files, noise_files)
    print(f"  Preprocessing: {(time.time()-t0)/60:.1f} min")

    if not event_data or not noise_data:
        print("ERROR: Could not load files."); return

    all_rows    = []
    patch_best  = {}
    t_total     = time.time()

    # ── [B3] TTFS — FIX 1 ────────────────────────────────────────────────────
    print("\n [1/5] TTFS  FIX 1: first-crossing (est. ~3 min)...")
    t0 = time.time()
    combos = [{'thr': t} for t in np.round(np.arange(0.40, 0.96, 0.05), 4)]
    print(f"  Grid: thr(12) = {len(combos)} combos")
    res = run_sweep_two_class('TTFS', combos, enc_ttfs,
                               event_data, noise_data, 'norm_data')
    patch_best['TTFS'] = report('TTFS', res, ['thr'], all_rows,
                                 '[FIX1: first-crossing]')
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── [P1] Delta-Mod — FIX 4 ───────────────────────────────────────────────
    print("\n [2/5] Delta-Mod  FIX 4: vectorised (est. ~5 min)...")
    t0 = time.time()
    thr_vals  = np.round(np.arange(0.40, 0.96, 0.05), 4)
    step_vals = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]
    combos    = [{'thr': t, 'step_size': ss}
                 for t in thr_vals for ss in step_vals]
    print(f"  Grid: thr(12) × step_size(11) = {len(combos)} combos")
    res = run_sweep_two_class('Delta-Mod', combos, enc_delta_mod,
                               event_data, noise_data, 'norm_data')
    patch_best['Delta-Mod'] = report('Delta-Mod', res,
                                      ['thr', 'step_size'], all_rows,
                                      '[FIX4: vectorised]')
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── [P5] ST-MW — FIX 5 ───────────────────────────────────────────────────
    print("\n [3/5] ST-MW  FIX 5: scipy spatial (est. ~15 min)...")
    t0 = time.time()
    thr_t_vals = [0.10, 0.14, 0.18, 0.22, 0.26, 0.30, 0.38, 0.46, 0.54, 0.62]
    thr_s_vals = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    wt_vals    = [0.5, 1.0, 2.0, 4.0]
    ws_vals    = [4, 8, 16]
    combos     = [{'thr_t': tt, 'thr_s': ts, 'wt_ms': wt, 'ws': ws}
                  for tt in thr_t_vals for ts in thr_s_vals
                  for wt in wt_vals for ws in ws_vals]
    print(f"  Grid: thr_t(10) × thr_s(8) × wt_ms(4) × ws(3) = {len(combos)} combos")
    res = run_sweep_two_class('ST-MW', combos, enc_stmw,
                               event_data, noise_data, 'norm_data')
    patch_best['ST-MW'] = report('ST-MW', res,
                                  ['thr_t', 'thr_s', 'wt_ms', 'ws'],
                                  all_rows, '[FIX5: scipy]')
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── [P6] CSS — NEW ───────────────────────────────────────────────────────
    print("\n [4/5] CSS  NEW — replaces ICDE (est. ~8 min)...")
    t0 = time.time()
    vth_vals = [0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
    mom_vals = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]
    combos   = [{'v_th': v, 'momentum': m}
                for v in vth_vals for m in mom_vals]
    print(f"  Grid: v_th(10) × momentum(6) = {len(combos)} combos")
    print(f"  Ternary {{+1,0,-1}} spikes | sparsity = |spikes|.mean()")
    res = run_sweep_two_class('CSS', combos, enc_css,
                               event_data, noise_data, 'norm_data')
    patch_best['CSS'] = report('CSS', res, ['v_th', 'momentum'],
                                all_rows, '[NEW: replaces ICDE]')
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── [P7] SFBE — FIX 3 ────────────────────────────────────────────────────
    print("\n [5/5] SFBE  FIX 3: STA/LTA (est. ~5 min)...")
    t0 = time.time()
    ratio_vals = [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 15.0]
    sta_vals   = [1.0, 2.0, 4.0, 8.0]
    combos     = [{'ratio_thr': r, 'sta_ms': s}
                  for r in ratio_vals for s in sta_vals]
    print(f"  Grid: ratio_thr(12) × sta_ms(4) = {len(combos)} combos")
    res = run_sweep_two_class('SFBE', combos, enc_sfbe,
                               event_data, noise_data, 'norm_data')
    patch_best['SFBE'] = report('SFBE', res, ['ratio_thr', 'sta_ms'],
                                 all_rows, '[FIX3: STA/LTA]')
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── Save patch CSV ────────────────────────────────────────────────────────
    csv_path = os.path.join(RESULTS_DIR, 'sweep_results_patch.csv')
    pd.DataFrame(all_rows).to_csv(csv_path, index=False)
    print(f"\n Patch results → {csv_path}")

    # ── Merge with v1 confirmed → single JSON ─────────────────────────────────
    skip = {'sparsity', 'sparsity_event', 'sparsity_noise',
            'sparsity_avg', 'snr', 'precision', 'in_target'}

    merged = {}

    # v1 confirmed (no re-run)
    for enc, v1 in V1_CONFIRMED.items():
        merged[enc] = {k: (list(v) if isinstance(v, list) else v)
                       for k, v in v1.items() if k not in skip}

    # patch results (new best)
    for enc, best in patch_best.items():
        if best is not None:
            merged[enc] = {
                k: (list(v) if isinstance(v, (list, np.ndarray)) else
                    float(v) if isinstance(v, (np.floating, float)) else v)
                for k, v in best.items() if k not in skip}
        else:
            merged[enc] = None

    json_path = os.path.join(RESULTS_DIR, 'best_params_patch.json')
    with open(json_path, 'w') as f:
        json.dump(merged, f, indent=2)
    print(f" Merged best params (v1 + patch) → {json_path}")

    # ── Final comparison table ────────────────────────────────────────────────
    enc_order = ['Rate','Phase','TTFS','Delta-Mod','PDE',
                 'ATDE','MASTE','ST-MW','CSS','SFBE']
    source_map = {
        'Rate':'v1✓','Phase':'v1✓','PDE':'v1✓','ATDE':'v1✓','MASTE':'v1✓',
        'TTFS':'patch','Delta-Mod':'patch','ST-MW':'patch',
        'CSS':'patch','SFBE':'patch',
    }
    fix_map = {
        'TTFS':      'FIX1:first-crossing',
        'Delta-Mod': 'FIX4:vectorised',
        'ST-MW':     'FIX5:scipy',
        'CSS':       'NEW:replaces ICDE',
        'SFBE':      'FIX3:STA/LTA',
    }

    print(f"\n\n{'='*100}")
    print(f"FINAL MERGED RESULTS — {N_SAMPLES} event + {N_SAMPLES} noise files")
    print(f"{'='*100}")
    print(f"  {'Encoder':>12}  {'Source':>7}  {'sp_ev%':>7}  {'sp_no%':>7}  "
          f"{'sp_avg%':>8}  {'SNR':>6}  {'Prec':>6}  Fix / Note")
    print(f"  {'─'*12}  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*8}  {'─'*6}  {'─'*6}  {'─'*25}")

    all_best = {**{k: v for k, v in V1_CONFIRMED.items()},
                **{k: v for k, v in patch_best.items() if v is not None}}

    for name in enc_order:
        src = source_map.get(name, '')
        fix = fix_map.get(name, '')
        b   = all_best.get(name)
        if b is None:
            print(f"  {name:>12}  {src:>7}  NO CONFIG IN TARGET  {fix}")
            continue
        sp_ev = b.get('sparsity_event', b.get('sparsity', 0))
        sp_no = b.get('sparsity_noise', b.get('sparsity', 0))
        sp_av = b.get('sparsity_avg',   b.get('sparsity', 0))
        snr   = b.get('snr', 0)
        prec  = b.get('precision', 0)
        print(f"  {name:>12}  {src:>7}  {sp_ev*100:>7.3f}  {sp_no*100:>7.3f}  "
              f"{sp_av*100:>8.3f}  {snr:>6.2f}  {prec:>6.3f}  {fix}")

    elapsed = time.time() - t_total
    print(f"\n  Patch sweep time: {elapsed/60:.1f} min")
    print(f"""
  Copy best_params_patch.json → 3__Preprocessing__10k_files_.ipynb
  This file contains ALL 10 encoders:
    5 from v1 (Rate, Phase, PDE, ATDE, MASTE) — not re-run
    5 from patch (TTFS, Delta-Mod, ST-MW, CSS, SFBE) — freshly swept
""")
    return merged


if __name__ == "__main__":
    run_patch_sweep()

In [ ]:
"""
MINI PATCH — CSS (Exponential) + SFBE (Fixed LTA)
==================================================
Fixes the 2 remaining failed encoders from the patch sweep.

PROBLEM RECAP:
  CSS  → sp=100% all combos. Momentum membrane V[t]=2×V[t-1]+x[t]
          explodes to infinity in vectorised form (resets need sequential loop).
          REPLACED with Exponential Coding (at-most-2-spike).

  SFBE → SNR=0.83× (noise fired MORE than events).
          Cumulative LTA contaminated by event energy → LTA grew large
          during event, then STA/LTA collapsed after event ended.
          FIXED with pre-event fixed baseline window (0–200ms).

NEW ENCODERS:
  [P6] ExpCod  — At-Most-Two-Spike Exponential Coding
                 Base: Neural Networks 2024
                 "High-performance deep SNNs via at-most-two-spike
                  exponential coding" (doi:10.1016/j.neunet.2024.106346)
                 DAS adaptation: per-trace amplitude normalisation +
                 exponential non-linearity separates P-wave vs S-wave
                 amplitude classes via second-spike threshold.

  [P7] SFBE    — Sub-band Frequency Band Encoding (FIXED LTA)
                 Base: FEEL-SNN NeurIPS 2024 (time-multiplex retained)
                 Fix: pre-event fixed baseline LTA (0–200ms window)
                 replaces cumulative LTA (was contaminated by event energy).

Estimated runtime:
  ExpCod: ~5  min  (80 combos, fully vectorised)
  SFBE:   ~8  min  (48 combos, scipy vectorised)
  TOTAL:  ~13 min

Output:
  sweep_results_minipatch.csv
  best_params_final.json   ← ALL 10 encoders (8 confirmed + 2 new)
"""

import os, json, time, random
import segyio
import numpy as np
import pandas as pd
from scipy.signal  import butter, filtfilt, hilbert

# =============================================================================
# CONFIGURATION
# =============================================================================
EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
RESULTS_DIR = config.SWEEP_RESULTS_DIR

N_SAMPLES   = 400
RANDOM_SEED = 42          # MUST match v1 — same files

FORCE_DT_MS     = 0.5
FILTER_LOW_HZ   = 50
FILTER_HIGH_HZ  = 250
WINDOW_START_MS = 0
WINDOW_END_MS   = 1000
EVENT_START_MS  = 380.0
EVENT_END_MS    = 560.0
PRE_EVENT_END_MS = 200.0  # fixed LTA baseline window end

TARGET_SP_LO = 0.003
TARGET_SP_HI = 0.015

SFBE_BANDS = [(50, 100), (100, 150), (150, 200), (200, 250)]

os.makedirs(RESULTS_DIR, exist_ok=True)

# =============================================================================
# 8 CONFIRMED BEST PARAMS (from v1 + patch sweeps — not re-run)
# =============================================================================
CONFIRMED_8 = {
    'Rate':      {'thr': 0.85,
                  'sparsity_event': 0.00409, 'sparsity_noise': 0.00514,
                  'sparsity_avg': 0.00461, 'snr': 3.09, 'precision': 0.253},
    'Phase':     {'thr': 0.85,
                  'sparsity_event': 0.00444, 'sparsity_noise': 0.00570,
                  'sparsity_avg': 0.00507, 'snr': 2.71, 'precision': 0.249},
    'TTFS':      {'thr': 0.85,
                  'sparsity_event': 0.00444, 'sparsity_noise': 0.00570,
                  'sparsity_avg': 0.00507, 'snr': 2.71, 'precision': 0.249,
                  'note': 'Converges to Phase on DAS sharp-onset data — reported as finding'},
    'Delta-Mod': {'thr': 0.85, 'step_size': 0.18,
                  'sparsity_event': 0.00269, 'sparsity_noise': 0.00339,
                  'sparsity_avg': 0.00304, 'snr': 2.63, 'precision': 0.251},
    'PDE':       {'kappa_phi': 0.5, 'amp_kappa': 3.0,
                  'sparsity_event': 0.00672, 'sparsity_noise': 0.00455,
                  'sparsity_avg': 0.00563, 'snr': 2.17, 'precision': 0.268},
    'ATDE':      {'alpha': 3.0, 'tau_ms': 165.0,
                  'sparsity_event': 0.00416, 'sparsity_noise': 0.00282,
                  'sparsity_avg': 0.00349, 'snr': 2.22, 'precision': 0.208},
    'MASTE':     {'lags': [1, 3, 8], 'alpha': 2.5,
                  'sparsity_event': 0.00366, 'sparsity_noise': 0.00256,
                  'sparsity_avg': 0.00311, 'snr': 2.52, 'precision': 0.264},
    'ST-MW':     {'thr_t': 0.18, 'thr_s': 0.45, 'wt_ms': 1.0, 'ws': 8,
                  'sparsity_event': 0.00504, 'sparsity_noise': 0.00609,
                  'sparsity_avg': 0.00556, 'snr': 5.40, 'precision': 0.283},
}


# =============================================================================
# FILE SAMPLING + LOAD (identical to all previous sweeps)
# =============================================================================
def sample_files(directory, n, seed):
    all_files = sorted([
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(('.sgy', '.segy'))
    ])
    if len(all_files) < n:
        return all_files
    return sorted(random.Random(seed).sample(all_files, n))


def load_and_preprocess(filepath):
    try:
        with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))],
                            dtype=np.float64)
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                dt_ms = dt_ms if 0.001 <= dt_ms <= 10.0 else FORCE_DT_MS
            except Exception:
                dt_ms = FORCE_DT_MS
    except Exception as ex:
        print(f"    ⚠️  {os.path.basename(filepath)}: {ex}")
        return None

    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ/nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ/nyq, 1e-6, 0.99)], btype='band')
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        filtered[i] = filtfilt(b, a, data[i])

    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(filtered.shape[1], int(WINDOW_END_MS / dt_ms))
    filtered = filtered[:, s:e]

    tmp = filtered - filtered.mean(axis=1, keepdims=True)
    mx  = np.where(np.max(np.abs(tmp), axis=1, keepdims=True) > 0,
                   np.max(np.abs(tmp), axis=1, keepdims=True), 1.0)
    norm_data = (tmp / mx + 1) / 2

    return {'norm_data': norm_data, 'filtered': filtered, 'dt_ms': dt_ms}


def load_dataset(event_files, noise_files):
    print(f"\n  Loading {len(event_files)} event files...")
    ev = [r for fp in event_files
          if (r := load_and_preprocess(fp)) is not None]
    print(f"  Loading {len(noise_files)} noise files...")
    no = [r for fp in noise_files
          if (r := load_and_preprocess(fp)) is not None]
    print(f"  Loaded: {len(ev)} event + {len(no)} noise")
    return ev, no


# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(spikes, dt_ms):
    n_tr, n_smp = spikes.shape
    sp  = float(spikes.mean())
    s   = max(0, int(EVENT_START_MS / dt_ms))
    e   = min(n_smp, int(EVENT_END_MS / dt_ms))
    density = spikes.mean(axis=0)
    in_ev   = float(density[s:e].mean()) if e > s else 0.0
    out_smp = np.concatenate([density[:s], density[e:]])
    min_d   = 1.0 / max(n_tr * n_smp, 1)
    out_ev  = float(max(out_smp.mean(), min_d)) if len(out_smp) > 0 else min_d
    snr     = in_ev / out_ev if out_ev > 0 else 0.0
    total   = float(spikes.sum())
    prec    = float(spikes[:, s:e].sum() / total) if total > 0 else 0.0
    return sp, snr, prec


def batch_metrics(spikes_list, dt_ms_list):
    sps, snrs, precs = [], [], []
    for spk, dt in zip(spikes_list, dt_ms_list):
        sp, snr, prec = compute_metrics(spk, dt)
        sps.append(sp); snrs.append(snr); precs.append(prec)
    return float(np.mean(sps)), float(np.mean(snrs)), float(np.mean(precs))


# =============================================================================
# NEW ENCODER 1 — ExpCod (Exponential At-Most-Two-Spike)
# =============================================================================
def enc_expcod(data, dt, thr1, kappa, thr2_ratio=0.5):
    """
    P6: Exponential At-Most-Two-Spike Encoding (ExpCod).
    Base: "High-performance deep SNNs via at-most-two-spike exponential
           coding", Neural Networks 2024 (doi:10.1016/j.neunet.2024.106346)

    Original algorithm (ANN-SNN conversion):
      Each ANN activation a is encoded into at most 2 spikes using:
        Spike 1 at step t=1 if a > threshold_1
        Spike 2 at step t=2 if a - exp(-kappa × a) > threshold_2
      The exponential term introduces NON-LINEARITY — unlike Rate/Phase
      which have linear amplitude-to-spike mappings, ExpCod compresses
      large activations (logarithmic scale) and spreads small ones.

    DAS adaptation:
      1. Amplitude computed per time window (not per ANN layer activation)
      2. thr1 scales with per-trace normalised amplitude [0,1]
      3. Spike 1 fires at the PEAK SAMPLE within the window (onset coding)
      4. Spike 2 fires at the MIDPOINT if exponential residual > thr2
         → Second spike only for high-energy P-wave arrivals
         → One spike only for moderate S-wave / surface wave arrivals
         → No spikes for background noise
      5. At most 2 spikes per window → sparsity naturally bounded

    Physical DAS interpretation:
      The exponential function exp(-kappa × amp) acts as a compressor:
        High amp (strong P-wave): residual = amp - exp(-kappa × amp) ≈ amp → LARGE
                                  → second spike fires → SNN gets 2 spikes
        Low amp  (weak S-wave):   residual = amp - exp(-kappa × amp) ≈ 0  → SMALL
                                  → no second spike → SNN gets 1 spike
        Noise:   amp < thr1       → no spikes at all

    Parameters swept:
      thr1:  [0.40 → 0.90] first-spike amplitude threshold (10 values)
      kappa: [0.5  → 8.0]  exponential compression factor  (8 values)
      thr2_ratio: fixed at 0.5 (second-spike fires when residual > 0.5 × thr1)

    Total: 80 combos. Fully vectorised — no sequential membrane loop.
    """
    spw    = max(1, int(1.0 / dt))
    n_tr, n_smp = data.shape
    n_bins = n_smp // spw
    trimmed = data[:, :n_bins * spw]
    binned  = trimmed.reshape(n_tr, n_bins, spw)
    abs_b   = np.abs(binned)

    # ── Peak amplitude and position per window ────────────────────────────────
    amp      = abs_b.max(axis=2)                          # (n_tr, n_bins)
    peak_off = np.argmax(abs_b, axis=2)                   # (n_tr, n_bins)
    starts   = np.arange(n_bins) * spw

    # ── Spike 1: peak sample if amp > thr1 ───────────────────────────────────
    mask1   = amp > thr1
    idx1    = np.clip(starts[np.newaxis, :] + peak_off, 0, n_smp - 1)

    # ── Spike 2: midpoint if exponential residual > thr2 ─────────────────────
    # residual = amp - exp(-kappa × amp)   [non-linear amplitude transform]
    residual = amp - np.exp(-kappa * amp)
    thr2     = thr2_ratio * thr1
    mask2    = mask1 & (residual > thr2)                  # only where spike1 also fired
    idx2     = np.clip(starts[np.newaxis, :] + spw // 2,  # midpoint of window
                       0, n_smp - 1)

    # ── Write output ──────────────────────────────────────────────────────────
    out    = np.zeros((n_tr, n_smp), dtype=np.float32)
    tr_idx = np.repeat(np.arange(n_tr)[:, np.newaxis], n_bins, axis=1)
    out[tr_idx[mask1], idx1[mask1]] = 1.0
    out[tr_idx[mask2], idx2[mask2]] = 1.0           # may overlap with spike1; fine
    return out


# =============================================================================
# NEW ENCODER 2 — SFBE with Fixed Pre-Event Baseline LTA
# =============================================================================
def enc_sfbe_fixed(data, dt, ratio_thr, sta_ms, bands=None):
    """
    P7: Sub-band Frequency Band Encoding — Fixed Baseline LTA.
    Base: FEEL-SNN NeurIPS 2024 (time-multiplexing structure retained).

    WHY THE PREVIOUS VERSION FAILED (SNR=0.83×, noise > events):
      Cumulative LTA[t] = mean(env²[0:t]) grows large during the event
      window (380–560ms) because the high event energy inflates the
      cumulative mean. After the event ends, LTA stays elevated while
      STA drops back to noise floor → STA/LTA collapses < ratio_thr
      exactly in the region AFTER the event. Meanwhile, noise files
      have no such inflation → random transients cause STA/LTA spikes
      → noise fires MORE than events → SNR < 1.0.

    THE FIX — Pre-event fixed baseline LTA:
      Use ONLY the first 0–200ms as the noise baseline.
      This window is ALWAYS before the event (event starts at 380ms).
      LTA is computed once per trace as the mean energy in 0–200ms.
      LTA is then FIXED (scalar per trace, per band) for the full trace.

      Fire when STA[t] / LTA_fixed > ratio_thr

    Why this works:
      Event files:  LTA_fixed = quiet pre-event noise level (small)
                    During event: STA spikes to 10–50× LTA_fixed → fires
                    Outside event: STA ≈ LTA_fixed → ratio ≈ 1 → no spike
      Noise files:  LTA_fixed = similar noise level
                    No event → STA stays near LTA_fixed → ratio ≈ 1 → no spike
                    Only random transients briefly exceed ratio_thr

    Time-multiplexing retained from FEEL-SNN:
      Band 3 (150–200 Hz, P-wave) fires early in event window
      Band 1 (50–100 Hz,  S-wave) fires later in event window
      SNN temporal dimension encodes wave type via band position

    Parameters swept:
      ratio_thr: [2.0 → 20.0] STA/LTA detection threshold (12 values)
      sta_ms:    [1, 2, 4, 8] ms short-term averaging window (4 values)
    Total: 48 combos.
    """
    if bands is None:
        bands = SFBE_BANDS
    fs_hz   = 1000.0 / dt
    n_tr, n_smp = data.shape
    n_bands = len(bands)
    seg_len = n_smp // n_bands
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    sta_w   = max(1, int(sta_ms / dt))

    # ── Pre-event baseline window (0–200ms, ALWAYS before event) ─────────────
    pre_end_smp = max(1, int(PRE_EVENT_END_MS / dt))  # sample index of 200ms

    for b_idx, (lo, hi) in enumerate(bands):
        nyq   = 0.5 * fs_hz
        blow  = np.clip(lo / nyq, 1e-6, 0.99)
        bhigh = np.clip(hi / nyq, 1e-6, 0.99)
        b_c, a_c = butter(4, [blow, bhigh], btype='band')

        # Filter + squared envelope
        filt_b = np.zeros_like(data)
        for t in range(n_tr):
            filt_b[t] = filtfilt(b_c, a_c, data[t])
        env_sq = np.abs(hilbert(filt_b, axis=1)) ** 2    # (n_tr, n_smp)

        # ── Fixed LTA: mean energy in pre-event window per trace ──────────────
        lta_fixed = env_sq[:, :pre_end_smp].mean(axis=1, keepdims=True)
        lta_fixed = np.maximum(lta_fixed, 1e-12)          # avoid division by zero

        # ── STA: causal moving average (same as before) ───────────────────────
        cs  = np.cumsum(env_sq, axis=1)
        sta = np.zeros_like(env_sq)
        sta[:, :sta_w] = cs[:, :sta_w] / np.arange(1, sta_w + 1)
        sta[:, sta_w:] = (cs[:, sta_w:] - cs[:, :-sta_w]) / sta_w

        # ── Fire where STA / LTA_fixed > ratio_thr ───────────────────────────
        ratio = sta / lta_fixed                           # (n_tr, n_smp)
        fire  = ratio > ratio_thr
        fire[:, :sta_w] = False                           # warm-up

        # ── Time-multiplex: map fire → output segment for this band ───────────
        seg_start = b_idx * seg_len
        seg_end   = seg_start + seg_len if b_idx < n_bands - 1 else n_smp
        tr_ii, t_ii = np.where(fire)
        seg_pos = seg_start + (t_ii * seg_len // n_smp)
        valid   = ((seg_pos >= seg_start) & (seg_pos < seg_end)
                   & (seg_pos < n_smp))
        out[tr_ii[valid], seg_pos[valid]] = 1.0

    return out


# =============================================================================
# SWEEP RUNNER
# =============================================================================
def run_sweep(name, param_combos, encode_fn, event_data, noise_data,
              key='norm_data'):
    results  = []
    n_combos = len(param_combos)
    for ci, params in enumerate(param_combos):
        ev_spk, ev_dt = [], []
        for rec in event_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                ev_spk.append(spk); ev_dt.append(rec['dt_ms'])
            except Exception:
                continue
        no_spk = []
        for rec in noise_data:
            try:
                spk = encode_fn(rec[key], rec['dt_ms'], **params)
                no_spk.append(spk)
            except Exception:
                continue
        if not ev_spk:
            continue
        ev_sp, ev_snr, ev_prec = batch_metrics(ev_spk, ev_dt)
        no_sp  = float(np.mean([s.mean() for s in no_spk])) if no_spk else ev_sp
        sp_avg = (ev_sp + no_sp) / 2
        in_tgt = TARGET_SP_LO <= sp_avg <= TARGET_SP_HI
        results.append({**params,
                        'sparsity_event': ev_sp,
                        'sparsity_noise': no_sp,
                        'sparsity_avg':   sp_avg,
                        'snr':            ev_snr,
                        'precision':      ev_prec,
                        'in_target':      in_tgt})
        if (ci+1) % 4 == 0 or (ci+1) == n_combos:
            print(f"    [{ci+1:>3d}/{n_combos}]  "
                  f"sp_ev={ev_sp*100:.3f}%  sp_no={no_sp*100:.3f}%  "
                  f"sp_avg={sp_avg*100:.3f}%  snr={ev_snr:.2f}×  "
                  f"{'✓' if in_tgt else '✗'}",
                  flush=True)
    return results


def report(name, results, param_names):
    if not results:
        print(f"\n  {name}: no results"); return None
    target = [r for r in results if r['in_target']]
    target.sort(key=lambda x: x['snr'], reverse=True)
    print(f"\n  {'─'*76}")
    print(f"  {name}: {len(target)}/{len(results)} in target "
          f"({TARGET_SP_LO*100:.1f}%–{TARGET_SP_HI*100:.1f}%)")
    print(f"  {'─'*76}")
    show = target[:6] if target else sorted(
        results, key=lambda x: abs(x['sparsity_avg'] - 0.007))[:6]
    for r in show:
        vals = '  '.join(
            f'{r[p]:>9.4f}' if isinstance(r[p], float)
            else f'{str(r[p]):>9}' for p in param_names)
        print(f"  {vals}  sp_avg={r['sparsity_avg']*100:.3f}%  "
              f"snr={r['snr']:.2f}×  prec={r['precision']:.3f}  "
              f"{'★' if r['in_target'] else ' '}")
    best = target[0] if target else None
    if best:
        skip = {'sparsity','sparsity_event','sparsity_noise',
                'sparsity_avg','snr','precision','in_target'}
        rec  = ', '.join(f'{k}={v}' for k, v in best.items() if k not in skip)
        print(f"\n  → BEST: {rec}")
        print(f"    sp_ev={best['sparsity_event']*100:.3f}%  "
              f"sp_no={best['sparsity_noise']*100:.3f}%  "
              f"sp_avg={best['sparsity_avg']*100:.3f}%  "
              f"snr={best['snr']:.2f}×  prec={best['precision']:.3f}")
    else:
        closest = sorted(results,
                         key=lambda x: abs(x['sparsity_avg'] - 0.007))[0]
        skip = {'sparsity','sparsity_event','sparsity_noise',
                'sparsity_avg','snr','precision','in_target'}
        rec  = ', '.join(f'{k}={v}' for k, v in closest.items()
                         if k not in skip)
        print(f"  → No config in target. Closest: {rec}  "
              f"sp={closest['sparsity_avg']*100:.3f}%  "
              f"snr={closest['snr']:.2f}×")
    return best


# =============================================================================
# MAIN
# =============================================================================
def run_minipatch():
    print("=" * 70)
    print("MINI PATCH — ExpCod + SFBE (fixed LTA)")
    print(f"  Samples: {N_SAMPLES} event + {N_SAMPLES} noise = {2*N_SAMPLES} files")
    print(f"  Target:  {TARGET_SP_LO*100:.1f}% – {TARGET_SP_HI*100:.1f}%  sparsity")
    print(f"  Encoders: ExpCod (replaces CSS) | SFBE (fixed LTA)")
    print("=" * 70)

    print("\nSampling files (same seed as all previous sweeps)...")
    ev_files = sample_files(EVENT_DIR, N_SAMPLES, RANDOM_SEED)
    no_files = sample_files(NOISE_DIR, N_SAMPLES, RANDOM_SEED + 1)

    t0 = time.time()
    event_data, noise_data = load_dataset(ev_files, no_files)
    print(f"  Preprocessing: {(time.time()-t0)/60:.1f} min")

    if not event_data or not noise_data:
        print("ERROR: Could not load files."); return

    all_rows   = []
    new_best   = {}
    t_total    = time.time()

    # ── [P6] ExpCod ───────────────────────────────────────────────────────────
    # thr1:  0.40 → 0.90  (10 values)
    # kappa: 0.5  → 8.0   (8 values)
    # Total: 80 combos  ~5 min
    print("\n [1/2] ExpCod  (P6 — replaces CSS, Neural Networks 2024)...")
    t0 = time.time()
    thr1_vals  = np.round(np.arange(0.40, 0.91, 0.05), 2)
    kappa_vals = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
    combos     = [{'thr1': t, 'kappa': k}
                  for t in thr1_vals for k in kappa_vals]
    print(f"  Grid: thr1(10) × kappa(8) = {len(combos)} combos")
    print(f"  At-most-2-spike | exponential non-linearity | fully vectorised")
    res = run_sweep('ExpCod', combos, enc_expcod,
                    event_data, noise_data, 'norm_data')
    new_best['ExpCod'] = report('ExpCod', res, ['thr1', 'kappa'])
    for r in res:
        all_rows.append({'encoder': 'ExpCod', **r})
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── [P7] SFBE Fixed LTA ───────────────────────────────────────────────────
    # ratio_thr: 2.0 → 30.0  (12 values — wider to find the right operating point)
    # sta_ms:    1, 2, 4, 8  (4 values)
    # Total: 48 combos  ~8 min
    print("\n [2/2] SFBE  (P7 — fixed pre-event baseline LTA)...")
    t0 = time.time()
    ratio_vals = [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0,
                  10.0, 12.0, 15.0, 20.0, 30.0]
    sta_vals   = [1.0, 2.0, 4.0, 8.0]
    combos     = [{'ratio_thr': r, 'sta_ms': s}
                  for r in ratio_vals for s in sta_vals]
    print(f"  Grid: ratio_thr(12) × sta_ms(4) = {len(combos)} combos")
    print(f"  Fixed LTA baseline = 0–{PRE_EVENT_END_MS:.0f}ms "
          f"(pre-event window, always before 380ms event start)")
    print(f"  Sub-bands {SFBE_BANDS} | time-multiplex retained (FEEL-SNN)")
    res = run_sweep('SFBE', combos, enc_sfbe_fixed,
                    event_data, noise_data, 'norm_data')
    new_best['SFBE'] = report('SFBE', res, ['ratio_thr', 'sta_ms'])
    for r in res:
        all_rows.append({'encoder': 'SFBE', **r})
    print(f"  Time: {(time.time()-t0)/60:.1f} min")

    # ── Save CSV ──────────────────────────────────────────────────────────────
    csv_path = os.path.join(RESULTS_DIR, 'sweep_results_minipatch.csv')
    pd.DataFrame(all_rows).to_csv(csv_path, index=False)
    print(f"\n Mini-patch results → {csv_path}")

    # ── Merge all 10 into final JSON ──────────────────────────────────────────
    skip = {'sparsity','sparsity_event','sparsity_noise',
            'sparsity_avg','snr','precision','in_target','note'}
    final = {}

    # 8 confirmed (not re-run)
    for enc, v in CONFIRMED_8.items():
        final[enc] = {k: (list(vv) if isinstance(vv, list) else vv)
                      for k, vv in v.items() if k not in skip}

    # 2 new (from this sweep)
    for enc, best in new_best.items():
        if best is not None:
            final[enc] = {
                k: (list(v) if isinstance(v, (list, np.ndarray)) else
                    float(v) if isinstance(v, (np.floating, float)) else v)
                for k, v in best.items() if k not in skip}
        else:
            final[enc] = None

    json_path = os.path.join(RESULTS_DIR, 'best_params_final.json')
    with open(json_path, 'w') as f:
        json.dump(final, f, indent=2)
    print(f" FINAL best params (all 10 encoders) → {json_path}")

    # ── Final summary table — ALL 10 encoders ─────────────────────────────────
    enc_order = ['Rate','Phase','TTFS','Delta-Mod','PDE',
                 'ATDE','MASTE','ST-MW','ExpCod','SFBE']
    source_map = {
        'Rate':'v1','Phase':'v1','PDE':'v1','ATDE':'v1','MASTE':'v1',
        'TTFS':'patch','Delta-Mod':'patch','ST-MW':'patch',
        'ExpCod':'mini','SFBE':'mini',
    }

    all_results = {**CONFIRMED_8,
                   **{k: v for k, v in new_best.items() if v}}

    print(f"\n\n{'='*105}")
    print("FINAL PARAMS — ALL 10 ENCODERS  (v1 confirmed + patch + mini-patch)")
    print(f"{'='*105}")
    print(f"  {'Encoder':>12}  {'Src':>5}  {'sp_ev%':>7}  {'sp_no%':>7}  "
          f"{'sp_avg%':>8}  {'SNR':>6}  {'Prec':>6}  Parameters")
    print(f"  {'─'*12}  {'─'*5}  {'─'*7}  {'─'*7}  {'─'*8}  "
          f"{'─'*6}  {'─'*6}  {'─'*35}")

    for name in enc_order:
        src  = source_map.get(name, '')
        best = all_results.get(name)
        if best is None:
            print(f"  {name:>12}  {src:>5}  ─── NO CONFIG IN TARGET RANGE ───")
            continue
        sp_ev = best.get('sparsity_event', best.get('sparsity', 0))
        sp_no = best.get('sparsity_noise', best.get('sparsity', 0))
        sp_av = best.get('sparsity_avg',   best.get('sparsity', 0))
        snr   = best.get('snr', 0)
        prec  = best.get('precision', 0)
        params_str = ', '.join(
            f'{k}={v}' for k, v in best.items()
            if k not in {'sparsity','sparsity_event','sparsity_noise',
                         'sparsity_avg','snr','precision','in_target','note'})
        print(f"  {name:>12}  {src:>5}  {sp_ev*100:>7.3f}  {sp_no*100:>7.3f}  "
              f"{sp_av*100:>8.3f}  {snr:>6.2f}  {prec:>6.3f}  {params_str}")

    elapsed = time.time() - t_total
    print(f"\n  Mini-patch time: {elapsed/60:.1f} min")
    print(f"""
  ──────────────────────────────────────────────────────────────────────────
  KEY FINDINGS FROM ALL SWEEPS:
  ──────────────────────────────────────────────────────────────────────────
  ST-MW   SNR=5.40×  Best encoder overall — spatio-temporal wavefront gate
  Rate    SNR=3.09×  Strong baseline — confirms DAS events have clear SNR
  MASTE   SNR=2.52×  Multi-timescale voting effective on DAS transients
  ATDE    SNR=2.22×  Adaptive noise floor tracks DAS non-stationary noise
  TTFS = Phase       Converges on DAS — sharp-onset data finding (paper note)

  COPY best_params_final.json → 3__Preprocessing__10k_files_.ipynb
  This is the FINAL parameter set for batch encoding of the full dataset.
  ──────────────────────────────────────────────────────────────────────────
""")
    return final


if __name__ == "__main__":
    run_minipatch()

In [ ]:
"""
AMSTE SWEEP — Adaptive Multi-Scale Spatio-Temporal Delta Encoder
================================================================
Adds the 10th and final encoder to the benchmark.

AMSTE: Adaptive Multi-Scale Spatio-Temporal Delta Encoder
  PhD novelty encoder combining 4 DAS-specific improvements:

  1. Channel-adaptive MAD threshold  (vs fixed or global-EMA threshold)
     θ_c = α × 1.4826 × MAD(X_c)
     Each channel gets its own noise floor — handles DAS coupling variation.

  2. Bidirectional polarity spikes  (vs positive-only delta coding)
     +spike when ΔX > +θ_c   (compression wave — positive strain)
     −spike when ΔX < −θ_c   (rarefaction wave — negative strain)
     Preserves DAS strain-rate polarity structure.

  3. Multi-scale temporal voting  (extends MASTE with polarity)
     Lags [1, 4, 8] samples = [0.5, 2.0, 4.0 ms]
     Candidate spike only when ≥ min_votes lags agree (per polarity)
     Captures both sharp P-wave arrivals and longer S-wave codas.

  4. Spatial coherence gate  (extends ST-MW to post-vote filtering)
     Spike survives only when spatial amplitude variation across
     ws neighbouring channels exceeds threshold θ_s.
     Suppresses isolated single-channel noise.

  Output: binary {0,1} — compatible with all downstream SNN notebooks.
  A spike fires at (c,t) when multi-scale votes AND spatial gate agree.

  Reference design basis:
    - Channel-adaptive threshold: PhaseNet-DAS (Nature Comms 2023)
    - Polarity coding: Neural Coding in SNNs (Frontiers Neurosci 2021)
    - Multi-scale delta: Sensor SNN evaluation (arXiv 2407.09260 2024)
    - Spatial coherence: ST-MW (this work, extending Petro TNNLS 2020)

9 CONFIRMED ENCODERS (not re-run — hardcoded from all sweeps):
  Rate:      thr=0.85                    SNR=3.09×  prec=0.253
  Phase:     thr=0.85                    SNR=2.71×  prec=0.249
  TTFS:      thr=0.85                    SNR=2.71×  prec=0.249
  Delta-Mod: thr=0.85, step_size=0.18   SNR=2.63×  prec=0.251
  PDE:       kappa_phi=0.5, amp_k=3.0   SNR=2.17×  prec=0.268
  ATDE:      alpha=3.0, tau_ms=165.0    SNR=2.22×  prec=0.208
  MASTE:     lags=[1,3,8], alpha=2.5    SNR=2.52×  prec=0.264
  ST-MW:     thr_t=0.18, thr_s=0.45,
             wt_ms=1.0, ws=8            SNR=5.40×  prec=0.283
  SFBE:      ratio_thr=10.0, sta_ms=8.0 SNR=0.83×  prec=0.152
             (sp_ev/sp_no=6.4× — evaluated via sparsity ratio)

Estimated runtime: ~20 min (756 combos, fully vectorised)

Output:
  sweep_results_amste.csv
  best_params_complete.json  ← ALL 10 encoders — copy to notebook 3
"""

import os, json, time, random
import segyio
import numpy as np
import pandas as pd
from scipy.signal  import butter, filtfilt, hilbert
from scipy.ndimage import maximum_filter1d, minimum_filter1d

# =============================================================================
# CONFIGURATION
# =============================================================================
EVENT_DIR   = config.RAW_EVENT_DIR
NOISE_DIR   = config.RAW_NOISE_DIR
RESULTS_DIR = config.SWEEP_RESULTS_DIR

N_SAMPLES   = 400
RANDOM_SEED = 42         # must match all previous sweeps

FORCE_DT_MS     = 0.5
FILTER_LOW_HZ   = 50
FILTER_HIGH_HZ  = 250
WINDOW_START_MS = 0
WINDOW_END_MS   = 1000
EVENT_START_MS  = 380.0
EVENT_END_MS    = 560.0

TARGET_SP_LO = 0.003
TARGET_SP_HI = 0.015

os.makedirs(RESULTS_DIR, exist_ok=True)

# =============================================================================
# 9 CONFIRMED BEST PARAMS (all previous sweeps — not re-run)
# =============================================================================
CONFIRMED_9 = {
    'Rate':      {'thr': 0.85,
                  'sparsity_event': 0.00409, 'sparsity_noise': 0.00514,
                  'sparsity_avg': 0.00461, 'snr': 3.09, 'precision': 0.253},
    'Phase':     {'thr': 0.85,
                  'sparsity_event': 0.00444, 'sparsity_noise': 0.00570,
                  'sparsity_avg': 0.00507, 'snr': 2.71, 'precision': 0.249},
    'TTFS':      {'thr': 0.85,
                  'sparsity_event': 0.00444, 'sparsity_noise': 0.00570,
                  'sparsity_avg': 0.00507, 'snr': 2.71, 'precision': 0.249,
                  'note': 'Converges to Phase on DAS sharp-onset data'},
    'Delta-Mod': {'thr': 0.85, 'step_size': 0.18,
                  'sparsity_event': 0.00269, 'sparsity_noise': 0.00339,
                  'sparsity_avg': 0.00304, 'snr': 2.63, 'precision': 0.251},
    'PDE':       {'kappa_phi': 0.5, 'amp_kappa': 3.0,
                  'sparsity_event': 0.00672, 'sparsity_noise': 0.00455,
                  'sparsity_avg': 0.00563, 'snr': 2.17, 'precision': 0.268},
    'ATDE':      {'alpha': 3.0, 'tau_ms': 165.0,
                  'sparsity_event': 0.00416, 'sparsity_noise': 0.00282,
                  'sparsity_avg': 0.00349, 'snr': 2.22, 'precision': 0.208},
    'MASTE':     {'lags': [1, 3, 8], 'alpha': 2.5,
                  'sparsity_event': 0.00366, 'sparsity_noise': 0.00256,
                  'sparsity_avg': 0.00311, 'snr': 2.52, 'precision': 0.264},
    'ST-MW':     {'thr_t': 0.18, 'thr_s': 0.45, 'wt_ms': 1.0, 'ws': 8,
                  'sparsity_event': 0.00504, 'sparsity_noise': 0.00609,
                  'sparsity_avg': 0.00556, 'snr': 5.40, 'precision': 0.283},
    'SFBE':      {'ratio_thr': 10.0, 'sta_ms': 8.0,
                  'sparsity_event': 0.02373, 'sparsity_noise': 0.00370,
                  'sparsity_avg': 0.01372, 'snr': 0.83, 'precision': 0.152,
                  'note': 'SNR metric inapplicable to time-multiplexed output; '
                          'sp_ev/sp_no=6.4x is the discriminability metric'},
}


# =============================================================================
# FILE SAMPLING + LOAD
# =============================================================================
def sample_files(directory, n, seed):
    all_files = sorted([
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(('.sgy', '.segy'))
    ])
    if len(all_files) < n:
        return all_files
    return sorted(random.Random(seed).sample(all_files, n))


def load_and_preprocess(filepath):
    try:
        with segyio.open(filepath, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))],
                            dtype=np.float64)
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                dt_ms = dt_ms if 0.001 <= dt_ms <= 10.0 else FORCE_DT_MS
            except Exception:
                dt_ms = FORCE_DT_MS
    except Exception as ex:
        print(f"    ⚠️  {os.path.basename(filepath)}: {ex}")
        return None

    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(FILTER_LOW_HZ/nyq, 1e-6, 0.99),
                      np.clip(FILTER_HIGH_HZ/nyq, 1e-6, 0.99)], btype='band')
    filt = np.zeros_like(data)
    for i in range(data.shape[0]):
        filt[i] = filtfilt(b, a, data[i])

    s = max(0, int(WINDOW_START_MS / dt_ms))
    e = min(filt.shape[1], int(WINDOW_END_MS / dt_ms))
    filt = filt[:, s:e]

    tmp = filt - filt.mean(axis=1, keepdims=True)
    mx  = np.where(np.max(np.abs(tmp), axis=1, keepdims=True) > 0,
                   np.max(np.abs(tmp), axis=1, keepdims=True), 1.0)
    norm_data = (tmp / mx + 1) / 2

    return {'norm_data': norm_data, 'dt_ms': dt_ms}


def load_dataset(event_files, noise_files):
    print(f"\n  Loading {len(event_files)} event files...")
    ev = []
    for i, fp in enumerate(event_files):
        r = load_and_preprocess(fp)
        if r is not None:
            ev.append(r)
        if (i+1) % 100 == 0:
            print(f"    {i+1}/{len(event_files)}")

    print(f"  Loading {len(noise_files)} noise files...")
    no = []
    for i, fp in enumerate(noise_files):
        r = load_and_preprocess(fp)
        if r is not None:
            no.append(r)
        if (i+1) % 100 == 0:
            print(f"    {i+1}/{len(noise_files)}")

    print(f"  Loaded: {len(ev)} event + {len(no)} noise")
    return ev, no


# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(spikes, dt_ms):
    n_tr, n_smp = spikes.shape
    sp  = float(spikes.mean())
    s   = max(0, int(EVENT_START_MS / dt_ms))
    e   = min(n_smp, int(EVENT_END_MS / dt_ms))
    density = spikes.mean(axis=0)
    in_ev   = float(density[s:e].mean()) if e > s else 0.0
    out_smp = np.concatenate([density[:s], density[e:]])
    min_d   = 1.0 / max(n_tr * n_smp, 1)
    out_ev  = float(max(out_smp.mean(), min_d)) if len(out_smp) > 0 else min_d
    snr     = in_ev / out_ev if out_ev > 0 else 0.0
    total   = float(spikes.sum())
    prec    = float(spikes[:, s:e].sum() / total) if total > 0 else 0.0
    return sp, snr, prec


def batch_metrics(spikes_list, dt_ms_list):
    sps, snrs, precs = [], [], []
    for spk, dt in zip(spikes_list, dt_ms_list):
        sp, snr, prec = compute_metrics(spk, dt)
        sps.append(sp); snrs.append(snr); precs.append(prec)
    return float(np.mean(sps)), float(np.mean(snrs)), float(np.mean(precs))


# =============================================================================
# AMSTE ENCODER
# =============================================================================
def enc_amste(data, dt, alpha, lags, min_votes, ws, thr_s):
    """
    AMSTE: Adaptive Multi-Scale Spatio-Temporal Delta Encoder.

    Combines 4 DAS-specific improvements into one unified encoder:

    STEP 1 — Channel-Adaptive MAD Threshold
    ─────────────────────────────────────────
    For each channel c independently:
        MAD_c    = median(|X[c,:] - median(X[c,:])|)
        θ_c      = α × 1.4826 × MAD_c

    The 1.4826 factor makes MAD consistent with Gaussian σ.
    Each channel gets its own threshold calibrated to its local noise floor.
    This is the key improvement over ATDE (global EMA) and Delta-Mod (fixed):
    channels with poor coupling get a higher threshold automatically.

    Vectorised: median computed over all channels simultaneously.

    STEP 2 — Bidirectional Multi-Scale Temporal Differences
    ────────────────────────────────────────────────────────
    For each lag L in lags (e.g. [1, 4, 8] = [0.5, 2.0, 4.0 ms]):
        ΔX_L[c,t] = X[c,t] - X[c, t-L]

        pos_vote_L[c,t] = 1 if ΔX_L[c,t] >  +θ_c   (compression)
        neg_vote_L[c,t] = 1 if ΔX_L[c,t] <  -θ_c   (rarefaction)

    Captures DAS strain-rate polarity:
        P-wave: sharp positive then negative swing
        S-wave: lower-frequency oscillation
        Surface: long-duration, lower-amplitude

    STEP 3 — Multi-Scale Majority Vote (per polarity)
    ──────────────────────────────────────────────────
    pos_votes[c,t] = sum_L(pos_vote_L[c,t])
    neg_votes[c,t] = sum_L(neg_vote_L[c,t])

    candidate[c,t] = 1  if  pos_votes >= min_votes
                          OR neg_votes >= min_votes

    A real DAS seismic arrival is coherent across multiple timescales.
    Random noise spikes are impulsive (only lag-1) and fail the vote.
    This is identical in spirit to MASTE but applied per polarity separately
    rather than to unsigned differences.

    STEP 4 — Spatial Coherence Gate (vectorised)
    ─────────────────────────────────────────────
    At each candidate (c,t), check ws neighbouring channels:
        da_s[c,t] = max(|X[c-ws//2 : c+ws//2, t]|)
                    - min(|X[c-ws//2 : c+ws//2, t]|)

    Keep candidate spike only if da_s[c,t] > thr_s.

    A seismic wavefront changes amplitude across neighbours coherently.
    Isolated noise spikes (fading, erratic) fail the spatial gate.

    Vectorised using scipy.ndimage spatial filters — same approach as ST-MW.

    OUTPUT
    ──────
    Binary {0,1} spike matrix (n_tr, n_smp) — compatible with all
    downstream SNN notebooks. A spike at (c,t) represents that channel c
    detected a multi-scale polarity-consistent spatio-temporal event at t.
    """
    n_tr, n_smp = data.shape

    # ── Step 1: Channel-adaptive MAD threshold ────────────────────────────────
    med_c   = np.median(data, axis=1, keepdims=True)       # (n_tr, 1)
    mad_c   = np.median(np.abs(data - med_c),
                        axis=1, keepdims=True)              # (n_tr, 1)
    theta_c = np.maximum(alpha * 1.4826 * mad_c, 1e-9)    # (n_tr, 1)

    # ── Step 2 + 3: Multi-scale polarity vote ────────────────────────────────
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)

    for lag in lags:
        diff_l = data[:, lag:] - data[:, :-lag]            # (n_tr, n_smp-lag)
        pos_l  = (diff_l >  theta_c).astype(np.int32)
        neg_l  = (diff_l < -theta_c).astype(np.int32)
        pos_votes[:, lag:] += pos_l
        neg_votes[:, lag:] += neg_l

    # Candidate spike where either polarity reaches min_votes
    candidate = ((pos_votes >= min_votes) |
                 (neg_votes >= min_votes))                  # (n_tr, n_smp)

    # ── Step 4: Spatial coherence gate (vectorised) ───────────────────────────
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=ws,
                                 axis=0, mode='nearest')   # (n_tr, n_smp)
    spat_min = minimum_filter1d(abs_data, size=ws,
                                 axis=0, mode='nearest')   # (n_tr, n_smp)
    da_s     = spat_max - spat_min                         # (n_tr, n_smp)

    # Final: candidate AND spatial gate
    out = (candidate & (da_s > thr_s)).astype(np.float32)
    return out


# =============================================================================
# SWEEP RUNNER + REPORTING
# =============================================================================
def run_sweep(param_combos, event_data, noise_data):
    results  = []
    n_combos = len(param_combos)
    for ci, params in enumerate(param_combos):
        ev_spk, ev_dt = [], []
        for rec in event_data:
            try:
                spk = enc_amste(rec['norm_data'], rec['dt_ms'], **params)
                ev_spk.append(spk); ev_dt.append(rec['dt_ms'])
            except Exception as ex:
                print(f"      event file exception: {ex}")
                continue
        no_spk = []
        for rec in noise_data:
            try:
                spk = enc_amste(rec['norm_data'], rec['dt_ms'], **params)
                no_spk.append(spk)
            except Exception:
                continue
        if not ev_spk:
            continue
        ev_sp, ev_snr, ev_prec = batch_metrics(ev_spk, ev_dt)
        no_sp  = float(np.mean([s.mean() for s in no_spk])) if no_spk else ev_sp
        sp_avg = (ev_sp + no_sp) / 2
        in_tgt = TARGET_SP_LO <= sp_avg <= TARGET_SP_HI
        # Sparsity discrimination ratio (sp_ev / sp_no) as secondary metric
        sp_ratio = ev_sp / max(no_sp, 1e-9)
        results.append({
            'alpha':      params['alpha'],
            'lags':       str(params['lags']),   # serialisable string for CSV
            'min_votes':  params['min_votes'],
            'ws':         params['ws'],
            'thr_s':      params['thr_s'],
            'sparsity_event': ev_sp,
            'sparsity_noise': no_sp,
            'sparsity_avg':   sp_avg,
            'sp_ratio':       sp_ratio,
            'snr':            ev_snr,
            'precision':      ev_prec,
            'in_target':      in_tgt,
        })
        if (ci+1) % 10 == 0 or (ci+1) == n_combos:
            print(f"    [{ci+1:>3d}/{n_combos}]  "
                  f"sp_ev={ev_sp*100:.3f}%  sp_no={no_sp*100:.3f}%  "
                  f"sp_avg={sp_avg*100:.3f}%  "
                  f"ratio={sp_ratio:.2f}×  snr={ev_snr:.2f}×  "
                  f"{'✓' if in_tgt else '✗'}",
                  flush=True)
    return results


def report_amste(results):
    if not results:
        print("  AMSTE: no results"); return None

    target = [r for r in results if r['in_target']]
    # Primary sort: SNR (event discrimination)
    # Secondary sort: sp_ratio (overall separability)
    target.sort(key=lambda x: (x['snr'], x['sp_ratio']), reverse=True)

    print(f"\n  {'─'*80}")
    print(f"  AMSTE: {len(target)}/{len(results)} configs in target "
          f"({TARGET_SP_LO*100:.1f}%–{TARGET_SP_HI*100:.1f}%)")
    print(f"  {'─'*80}")
    print(f"  {'alpha':>7}  {'lags':>14}  {'mv':>3}  {'ws':>4}  "
          f"{'thr_s':>6}  sp_avg%  ratio×  snr×  prec")

    show = target[:8] if target else sorted(
        results, key=lambda x: abs(x['sparsity_avg'] - 0.007))[:8]
    for r in show:
        tag = '★' if r['in_target'] else ' '
        print(f"  {r['alpha']:>7.2f}  {r['lags']:>14}  "
              f"{r['min_votes']:>3}  {r['ws']:>4}  {r['thr_s']:>6.2f}  "
              f"{r['sparsity_avg']*100:>6.3f}  {r['sp_ratio']:>6.2f}  "
              f"{r['snr']:>5.2f}  {r['precision']:.3f}  {tag}")

    best = target[0] if target else None
    if best:
        print(f"\n  → BEST:")
        print(f"    alpha={best['alpha']}, lags={best['lags']}, "
              f"min_votes={best['min_votes']}, ws={best['ws']}, "
              f"thr_s={best['thr_s']}")
        print(f"    sp_ev={best['sparsity_event']*100:.3f}%  "
              f"sp_no={best['sparsity_noise']*100:.3f}%  "
              f"sp_avg={best['sparsity_avg']*100:.3f}%  "
              f"ratio={best['sp_ratio']:.2f}×  "
              f"snr={best['snr']:.2f}×  prec={best['precision']:.3f}")
    else:
        closest = sorted(results,
                         key=lambda x: abs(x['sparsity_avg'] - 0.007))[0]
        print(f"  → No config in target. Closest:")
        print(f"    alpha={closest['alpha']}, lags={closest['lags']}, "
              f"min_votes={closest['min_votes']}, ws={closest['ws']}, "
              f"thr_s={closest['thr_s']}")
        print(f"    sp_avg={closest['sparsity_avg']*100:.3f}%  "
              f"snr={closest['snr']:.2f}×")
    return best


# =============================================================================
# MAIN
# =============================================================================
def run_amste_sweep():
    print("=" * 70)
    print("AMSTE SWEEP — Adaptive Multi-Scale Spatio-Temporal Delta Encoder")
    print(f"  Samples: {N_SAMPLES} event + {N_SAMPLES} noise = {2*N_SAMPLES} files")
    print(f"  Target:  {TARGET_SP_LO*100:.1f}% – {TARGET_SP_HI*100:.1f}%  sparsity")
    print("  4 DAS improvements: MAD-threshold | polarity | multi-scale | spatial")
    print("=" * 70)

    print("\nSampling files (same seed as all previous sweeps)...")
    ev_files = sample_files(EVENT_DIR, N_SAMPLES, RANDOM_SEED)
    no_files = sample_files(NOISE_DIR, N_SAMPLES, RANDOM_SEED + 1)

    t0 = time.time()
    event_data, noise_data = load_dataset(ev_files, no_files)
    print(f"  Preprocessing: {(time.time()-t0)/60:.1f} min")

    if not event_data or not noise_data:
        print("ERROR: Could not load files."); return

    # ── Parameter Grid ─────────────────────────────────────────────────────────
    # alpha:     MAD multiplier — 1.0 (sensitive) to 4.0 (conservative)
    # lags:      4 combinations covering short/medium/long DAS timescales
    # min_votes: 1 (any scale fires) / 2 (majority) / 3 (all agree)
    # ws:        spatial window in channels (16m, 32m, 64m aperture at 4m spacing)
    # thr_s:     spatial amplitude gate (0.20 easy / 0.35 moderate / 0.50 strict)
    #
    # Total: 7 × 4 × 3 × 3 × 3 = 756 combos
    alpha_vals = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
    lag_sets   = [
        [1, 4, 8],     # 0.5, 2.0, 4.0 ms — short/medium/long
        [1, 3, 8],     # 0.5, 1.5, 4.0 ms — MASTE best confirmed
        [1, 4, 12],    # 0.5, 2.0, 6.0 ms — extended long
        [2, 4, 8],     # 1.0, 2.0, 4.0 ms — even spacing
    ]
    min_votes_vals = [1, 2, 3]
    ws_vals        = [4, 8, 16]
    thr_s_vals     = [0.20, 0.35, 0.50]

    combos = [
        {'alpha': a, 'lags': lags,
         'min_votes': mv, 'ws': ws, 'thr_s': ts}
        for a    in alpha_vals
        for lags in lag_sets
        for mv   in min_votes_vals
        for ws   in ws_vals
        for ts   in thr_s_vals
    ]
    print(f"\n  Grid: alpha(7) × lags(4) × min_votes(3) × ws(3) × thr_s(3) "
          f"= {len(combos)} combos")
    print(f"  Fully vectorised (scipy spatial + numpy diff) — est. ~20 min\n")

    t0  = time.time()
    res = run_sweep(combos, event_data, noise_data)
    best = report_amste(res)
    elapsed = (time.time() - t0) / 60
    print(f"\n  AMSTE sweep time: {elapsed:.1f} min")

    # ── Save CSV ──────────────────────────────────────────────────────────────
    csv_path = os.path.join(RESULTS_DIR, 'sweep_results_amste.csv')
    pd.DataFrame(res).to_csv(csv_path, index=False)
    print(f" AMSTE results → {csv_path}")

    # ── Merge all 10 encoders into final JSON ─────────────────────────────────
    skip = {'sparsity', 'sparsity_event', 'sparsity_noise',
            'sparsity_avg', 'snr', 'precision', 'in_target',
            'sp_ratio', 'note'}

    final = {}
    # 9 confirmed
    for enc, v in CONFIRMED_9.items():
        final[enc] = {k: (list(vv) if isinstance(vv, list) else vv)
                      for k, vv in v.items() if k not in skip}

    # AMSTE (new)
    if best is not None:
        lags_raw = best['lags']
        lags_list = ([int(x) for x in lags_raw.strip('[]').split(',')]
                     if isinstance(lags_raw, str) else lags_raw)
        final['AMSTE'] = {
            'alpha':      float(best['alpha']),
            'lags':       lags_list,
            'min_votes':  int(best['min_votes']),
            'ws':         int(best['ws']),
            'thr_s':      float(best['thr_s']),
        }
    else:
        final['AMSTE'] = None

    json_path = os.path.join(RESULTS_DIR, 'best_params_complete.json')
    with open(json_path, 'w') as f:
        json.dump(final, f, indent=2)
    print(f" COMPLETE best params (all 10 encoders) → {json_path}")

    # ── Final table — ALL 10 encoders ─────────────────────────────────────────
    enc_order  = ['Rate','Phase','TTFS','Delta-Mod','PDE',
                  'ATDE','MASTE','ST-MW','SFBE','AMSTE']
    type_map   = {'Rate':'Baseline','Phase':'Baseline','TTFS':'Baseline',
                  'Delta-Mod':'Proposed','PDE':'Proposed','ATDE':'Proposed',
                  'MASTE':'Proposed','ST-MW':'Proposed',
                  'SFBE':'Proposed','AMSTE':'Proposed'}
    all_metrics = {}
    for enc, v in CONFIRMED_9.items():
        all_metrics[enc] = v
    if best is not None:
        all_metrics['AMSTE'] = {
            'sparsity_event': best['sparsity_event'],
            'sparsity_noise': best['sparsity_noise'],
            'sparsity_avg':   best['sparsity_avg'],
            'snr':            best['snr'],
            'precision':      best['precision'],
            'sp_ratio':       best['sp_ratio'],
        }

    print(f"\n\n{'='*105}")
    print("COMPLETE BENCHMARK — ALL 10 ENCODERS")
    print(f"{'='*105}")
    print(f"  {'Encoder':>12}  {'Type':>9}  {'sp_ev%':>7}  {'sp_no%':>7}  "
          f"{'sp_avg%':>8}  {'SNR':>6}  {'ratio×':>7}  {'Prec':>6}")
    print(f"  {'─'*12}  {'─'*9}  {'─'*7}  {'─'*7}  {'─'*8}  "
          f"{'─'*6}  {'─'*7}  {'─'*6}")

    for name in enc_order:
        etype = type_map.get(name, '')
        m     = all_metrics.get(name)
        if m is None:
            print(f"  {name:>12}  {etype:>9}  ─── NO CONFIG IN TARGET RANGE ───")
            continue
        sp_ev  = m.get('sparsity_event', 0)
        sp_no  = m.get('sparsity_noise', 0)
        sp_av  = m.get('sparsity_avg',   0)
        snr    = m.get('snr', 0)
        ratio  = m.get('sp_ratio', sp_ev / max(sp_no, 1e-9))
        prec   = m.get('precision', 0)
        # Flag best SNR and best ratio
        flag = ''
        if name == 'ST-MW': flag = ' ← best SNR'
        if name == 'SFBE':  flag = ' ← best ratio (time-multiplex)'
        if name == 'AMSTE' and best: flag = ' ← proposed DAS-ASE'
        print(f"  {name:>12}  {etype:>9}  {sp_ev*100:>7.3f}  {sp_no*100:>7.3f}  "
              f"{sp_av*100:>8.3f}  {snr:>6.2f}  {ratio:>7.2f}  "
              f"{prec:>6.3f}{flag}")

    print(f"""
  ──────────────────────────────────────────────────────────────────────────
  AMSTE NOVELTY STATEMENT (for paper):
  ──────────────────────────────────────────────────────────────────────────
  Existing SNN encoders (Rate, TTFS, Delta-Mod) were designed for images,
  ECG, EEG or simple 1D time-series. They do not account for DAS-specific
  properties: channel-dependent noise coupling, spatially coherent wavefront
  propagation, bidirectional strain polarity, and multi-scale event duration.

  AMSTE converts DAS time-channel panels into sparse binary spike events
  using: (1) per-channel MAD-adaptive thresholds, (2) bidirectional polarity
  detection across multiple temporal scales, and (3) a spatial neighbourhood
  coherence gate. This preserves arrival timing and propagation structure
  while suppressing channel-localised and stationary DAS noise.

  ──────────────────────────────────────────────────────────────────────────
  COPY best_params_complete.json → 3__Preprocessing__10k_files_.ipynb
  ──────────────────────────────────────────────────────────────────────────
""")
    return final


if __name__ == "__main__":
    run_amste_sweep()